# Complete MoE End-to-End Experimentation Notebook

This notebook provides a comprehensive end-to-end workflow for Mixture of Experts (MoE) model experimentation, including:
- Baseline dense model evaluation
- MoE model creation and evaluation
- Standard training with LoRA
- Knowledge Distillation training
- Multi-phase experiment orchestration for variant comparison
- Comprehensive metrics tracking with WandB and TensorBoard

## Environment Setup

In [1]:
import os

os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"
os.environ["TOKENIZERS_PARALLELISM"] = "false"

## Package Installation and Imports

In [2]:
import sys
import matplotlib
import seaborn
from collections import defaultdict
import psutil
import warnings

!{sys.executable} -m pip install -q kagglehub evaluate wandb tensorboard bitsandbytes accelerate peft transformers torch scikit-learn tqdm matplotlib seaborn pandas
!pip install --upgrade transformers

warnings.filterwarnings("ignore")

print("All libraries installed successfully!")

All libraries installed successfully!


## Weights & Biases Initialization

In [3]:
import os
from IPython import get_ipython

os.environ["WANDB_DISABLE_SERVICE"] = "true"
os.environ["WANDB_MODE"] = "disabled"

try:
    ipython = get_ipython()
    if ipython:
        events = ipython.events
        for event_name in ['pre_run_cell', 'post_run_cell']:
            if hasattr(events, 'callbacks') and event_name in events.callbacks:
                callbacks_to_remove = [
                    cb for cb in events.callbacks[event_name]
                    if hasattr(cb, '__self__') and 'wandb' in str(type(cb.__self__)).lower()
                ]
                for cb in callbacks_to_remove:
                    try:
                        events.callbacks[event_name].remove(cb)
                    except:
                        pass
except:
    pass

In [4]:
import os
import wandb

os.environ["WANDB_DISABLE_CODE"] = "true"
os.environ["WANDB_SILENT"] = "true"

_original_init = wandb.init
_original_log = wandb.log

def safe_init(*args, **kwargs):
    """Safely initialize wandb with error handling."""
    try:
        if wandb.run is not None:
            try:
                wandb.finish()
            except:
                pass
        
        kwargs.setdefault('settings', wandb.Settings())
        if isinstance(kwargs['settings'], wandb.Settings):
            kwargs['settings']._disable_stats = True
            kwargs['settings']._disable_meta = True
        
        return _original_init(*args, **kwargs)
    except (BrokenPipeError, ConnectionResetError, OSError) as e:
        print(f"Warning: wandb connection error (continuing without tracking): {type(e).__name__}")
        return None
    except Exception as e:
        print(f"Warning: wandb initialization error: {e}")
        return None

def safe_log(*args, **kwargs):
    """Safely log to wandb with error handling."""
    try:
        if wandb.run is not None:
            _original_log(*args, **kwargs)
    except (BrokenPipeError, ConnectionResetError, OSError):
        pass
    except Exception as e:
        pass

wandb.init = safe_init
wandb.log = safe_log

print("Wandb error handling enabled. Connection errors will be handled gracefully.")

Wandb error handling enabled. Connection errors will be handled gracefully.


In [5]:
import wandb

wandb.login()

wandb.init(
    project="MoE-variants!",
    config={
        "model": "mistralai/Mistral-7B-v0.1",
        "max_length": 512,
        "batch_size": 1,
        "gradient_accumulation_steps": 4,
        "epochs": 2,
        "learning_rate": 2e-5,
        "quantization": "4-bit",
        "lora_r": 8,
        "lora_alpha": 16,
    },
)

print("WandB initialized successfully")

WandB initialized successfully


## Dataset Download and Processing

In [6]:
import kagglehub

path = kagglehub.dataset_download("peiyuanliu2001/mmlu-dataset")

print("Dataset path:", path)

Dataset path: /root/.cache/kagglehub/datasets/peiyuanliu2001/mmlu-dataset/versions/2


In [7]:
import pandas as pd
from datasets import Dataset

df = pd.read_csv(f"{path}/train.csv")

print("Dataset Overview:")
print(f"  Total samples: {len(df):,}")
print(f"  Columns: {df.columns.tolist()}")
print(f"\nAnswer distribution:")
print(df['answer'].value_counts().to_dict())

print("\nSample questions:")
display(df.head())

Dataset Overview:
  Total samples: 98,487
  Columns: ['prompt', 'A', 'B', 'C', 'D', 'answer']

Answer distribution:
{'C': 26491, 'B': 25377, 'D': 24767, 'A': 21852}

Sample questions:


,prompt,A,B,C,D,answer
0,Davis decided to kill Adams. He set out for Ad...,Adams only.,Brooks only.,Case only.,Adams and Brooks,B
1,A state statute requires any person licensed t...,"guilty, because this is a public welfare offen...","guilty, because he cannot be excused on the ba...","not guilty, because the statute punishes omiss...","not guilty, because he was not aware of the va...",D
2,"Lender met Borrower on the street, demanded th...","Yes, because Mann threatened to use deadly for...","Yes, unless Mann was related to Borrower.","No, if it was apparent that Lender was about t...","No, because Lender was the original aggressor ...",C
3,Peter sued Don for breach of contract. The cou...,must permit Don to answer if he had objected t...,"may permit Don to answer, whether or not he ha...",may permit Don to answer only if he had object...,"cannot permit Don to answer, whether or not he...",B
4,Ames had painted Bell's house under a contract...,partial breach of contract only if Ames had pr...,partial breach of contract whether or not Ames...,total breach of contract only if Ames had prop...,total breach of contract whether or not Ames h...,C


In [8]:
from sklearn.model_selection import train_test_split

def format_mmlu_prompt(row):
    """Format MMLU questions for EVALUATION (no answer)."""
    prompt = f"""Question: {row['prompt']}

A. {row['A']}
B. {row['B']}
C. {row['C']}
D. {row['D']}

Answer:"""
    return prompt

def format_mmlu_prompt_with_answer(row):
    """Format MMLU questions for TRAINING (includes answer)."""
    prompt = f"""Question: {row['prompt']}

A. {row['A']}
B. {row['B']}
C. {row['C']}
D. {row['D']}

Answer: {row['answer']}"""
    return prompt

stratify_labels = df['subject'] if 'subject' in df.columns else None
train_indices, eval_indices = train_test_split(
    df.index,
    test_size=0.3,
    random_state=42,
    stratify=stratify_labels if stratify_labels is not None else None,
)

df['split'] = 'train'
df.loc[eval_indices, 'split'] = 'eval'
train_subset_for_examples = df.loc[train_indices].copy()

df['formatted_prompt'] = df.apply(format_mmlu_prompt, axis=1)
df['formatted_prompt_with_answer'] = df.apply(format_mmlu_prompt_with_answer, axis=1)

answer_to_idx = {'A': 0, 'B': 1, 'C': 2, 'D': 3}
df['answer_idx'] = df['answer'].map(answer_to_idx)

print("\nSample formatted prompt (EVALUATION - no answer):")
sample_prompt = df['formatted_prompt'].iloc[0]
print(sample_prompt[:500] + "..." if len(sample_prompt) > 500 else sample_prompt)

print(f"\nSample formatted prompt (TRAINING - with answer):")
sample_prompt_train = df['formatted_prompt_with_answer'].iloc[0]
print(sample_prompt_train[:550] + "..." if len(sample_prompt_train) > 550 else sample_prompt_train)

print(f"\nExpected answer: {df['answer'].iloc[0]}")

train_df = df[df['split'] == 'train'].copy()
eval_df = df[df['split'] == 'eval'].copy()

print(f"\nSplit summary:")
print(f"  Train samples: {len(train_df):,}")
print(f"  Eval samples:  {len(eval_df):,}")


Sample formatted prompt (EVALUATION - no answer):
Question: Davis decided to kill Adams. He set out for Adams's house. Before he got there he saw Brooks, who resembled Adams. Thinking that Brooks was Adams, Davis shot at Brooks. The shot missed Brooks but wounded Case, who was some distance away. Davis had not seen Case. In a prosecution under a statute that proscribes any attempt to commit murder, the district attorney should indicate that the intended victim(s) was/were

A. Adams only.
B. Brooks only.
C. Case only.
D. Adams and Brooks

Answer...

Sample formatted prompt (TRAINING - with answer):
Question: Davis decided to kill Adams. He set out for Adams's house. Before he got there he saw Brooks, who resembled Adams. Thinking that Brooks was Adams, Davis shot at Brooks. The shot missed Brooks but wounded Case, who was some distance away. Davis had not seen Case. In a prosecution under a statute that proscribes any attempt to commit murder, the district attorney should indicate that

In [9]:
from datasets import Dataset, DatasetDict

MAX_LENGTH = 512

columns_to_use = ['prompt', 'formatted_prompt', 'formatted_prompt_with_answer', 'answer', 'answer_idx']
if 'subject' in train_df.columns:
    columns_to_use.insert(0, 'subject')
if 'A' in train_df.columns:
    columns_to_use.extend(['A', 'B', 'C', 'D'])

train_dataset_raw = Dataset.from_pandas(
    train_df[columns_to_use],
    preserve_index=False,
)
eval_dataset_raw = Dataset.from_pandas(
    eval_df[columns_to_use],
    preserve_index=False,
)

dataset = DatasetDict({
    'train': train_dataset_raw,
    'test': eval_dataset_raw,
})
eval_dataset = eval_dataset_raw

print("Dataset prepared:")
print(f"  Training samples: {len(train_dataset_raw):,}")
print(f"  Evaluation samples: {len(eval_dataset_raw):,}")
print(f"  Columns: {eval_dataset_raw.column_names}")

Dataset prepared:
  Training samples: 68,940
  Evaluation samples: 29,547
  Columns: ['prompt', 'formatted_prompt', 'formatted_prompt_with_answer', 'answer', 'answer_idx', 'A', 'B', 'C', 'D']


## Model loading (8-bit quantized)

In [10]:
from transformers import AutoModelForCausalLM, BitsAndBytesConfig
import torch

model_id = "mistralai/Mistral-7B-v0.1"

bnb_config = BitsAndBytesConfig(
    load_in_8bit=True,
    bnb_8bit_use_double_quant=True,
    bnb_8bit_compute_dtype=torch.bfloat16,
    bnb_8bit_quant_type="nf4",
    llm_int8_enable_fp32_cpu_offload=True
)

device_map = {"": "cuda:0"}

print("Loading model with 8-bit quantization (QLoRA)...")

model = AutoModelForCausalLM.from_pretrained(
    model_id,
    quantization_config=bnb_config,
    device_map=device_map,
    trust_remote_code=True
)

base_model = model

Loading model with 8-bit quantization (QLoRA)...


Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

In [11]:
from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained(model_id, trust_remote_code=True)

tokenizer.pad_token = tokenizer.eos_token

In [12]:
ANSWER_TOKENS = {
    'A': tokenizer.encode('A', add_special_tokens=False)[0],
    'B': tokenizer.encode('B', add_special_tokens=False)[0],
    'C': tokenizer.encode('C', add_special_tokens=False)[0],
    'D': tokenizer.encode('D', add_special_tokens=False)[0],
}

print("Answer token IDs:")
for letter, token_id in ANSWER_TOKENS.items():
    decoded = tokenizer.decode([token_id])
    print(f"  {letter}: {token_id} -> '{decoded}'")

idx_to_letter = {0: 'A', 1: 'B', 2: 'C', 3: 'D'}

Answer token IDs:
  A: 330 -> 'A'
  B: 365 -> 'B'
  C: 334 -> 'C'
  D: 384 -> 'D'


## Evaluation functions

In [13]:
import numpy as np
from tqdm import tqdm
from sklearn.metrics import confusion_matrix
import time
import gc
import torch

def compute_ece(confidences, predictions, labels, n_bins=10):
    """
    Compute Expected Calibration Error (ECE).
    Measures how well model confidence aligns with actual accuracy.
    """
    bin_boundaries = np.linspace(0, 1, n_bins + 1)
    bin_lowers = bin_boundaries[:-1]
    bin_uppers = bin_boundaries[1:]

    ece = 0.0
    for bin_lower, bin_upper in zip(bin_lowers, bin_uppers):
        in_bin = (confidences > bin_lower) & (confidences <= bin_upper)
        prop_in_bin = in_bin.mean()

        if prop_in_bin > 0:
            accuracy_in_bin = (predictions[in_bin] == labels[in_bin]).mean()
            avg_confidence_in_bin = confidences[in_bin].mean()
            ece += np.abs(avg_confidence_in_bin - accuracy_in_bin) * prop_in_bin

    return ece

def evaluate_mmlu_comprehensive(model, tokenizer, eval_dataset, answer_tokens,
                                 device="cuda", max_samples=None, show_progress=True):
    """
    Comprehensive MMLU evaluation with all Priority 1-3 metrics.

    Returns dict with:
    - accuracy: Overall accuracy
    - top2_accuracy: Top-2 accuracy
    - confidences: Model confidence scores
    - predictions: Predicted answers
    - true_labels: Ground truth answers
    - confusion_matrix: Confusion matrix
    - ece: Expected Calibration Error
    - throughput: Samples per second
    - avg_latency: Average latency per sample
    """
    model.eval()

    correct = 0
    top2_correct = 0
    total = 0

    all_confidences = []
    all_predictions = []
    all_true_labels = []

    samples = eval_dataset
    if max_samples is not None:
        samples = eval_dataset.select(range(min(max_samples, len(eval_dataset))))

    answer_token_ids = torch.tensor([answer_tokens['A'], answer_tokens['B'],
                                      answer_tokens['C'], answer_tokens['D']])

    start_time = time.time()

    iterator = tqdm(samples, desc="Evaluating", disable=not show_progress)

    with torch.no_grad():
        for example in iterator:
            prompt = example['formatted_prompt']
            true_answer = example['answer']
            true_idx = answer_to_idx[true_answer]

            inputs = tokenizer(
                prompt,
                return_tensors="pt",
                truncation=True,
                max_length=MAX_LENGTH
            ).to(device)

            outputs = model(**inputs)
            last_logits = outputs.logits[0, -1, :]

            answer_logits = last_logits[answer_token_ids]
            answer_probs = torch.softmax(answer_logits, dim=0)

            pred_idx = answer_probs.argmax().item()
            pred_answer = idx_to_letter[pred_idx]
            confidence = answer_probs[pred_idx].item()

            top2_indices = answer_probs.topk(2).indices.tolist()

            all_confidences.append(confidence)
            all_predictions.append(pred_idx)
            all_true_labels.append(true_idx)

            if pred_answer == true_answer:
                correct += 1
            if true_idx in top2_indices:
                top2_correct += 1
            total += 1

    end_time = time.time()
    duration = end_time - start_time

    confidences = np.array(all_confidences)
    predictions = np.array(all_predictions)
    true_labels = np.array(all_true_labels)

    accuracy = correct / total if total > 0 else 0.0
    top2_accuracy = top2_correct / total if total > 0 else 0.0
    ece = compute_ece(confidences, predictions, true_labels)
    conf_matrix = confusion_matrix(true_labels, predictions, labels=[0, 1, 2, 3])

    throughput = total / duration if duration > 0 else 0.0
    avg_latency = duration / total if total > 0 else 0.0

    return {
        'accuracy': accuracy,
        'top2_accuracy': top2_accuracy,
        'correct': correct,
        'total': total,
        'ece': ece,
        'confidences': confidences,
        'predictions': predictions,
        'true_labels': true_labels,
        'confusion_matrix': conf_matrix,
        'throughput': throughput,
        'avg_latency': avg_latency,
    }

def compute_model_flops(model, seq_length=512):
    """
    Estimate FLOPs per forward pass.

    Formula: FLOPs ≈ 2 * active_params * seq_length
    For MoE: multiply by sparsity factor (active_experts / total_experts)
    """
    config = model.config
    n_layers = config.num_hidden_layers
    d_model = config.hidden_size
    intermediate_size = config.intermediate_size
    vocab_size = config.vocab_size

    attention_flops = 4 * seq_length * d_model * d_model * n_layers

    ffn_flops = 8 * seq_length * d_model * intermediate_size * n_layers

    total_flops = attention_flops + ffn_flops

    is_moe = False
    sparsity_factor = 1.0
    
    try:
        base_model = model
        if hasattr(model, 'base_model'):
            base_model = model.base_model
        if hasattr(base_model, 'model'):
            base_model = base_model.model
        
        layers = base_model.layers if hasattr(base_model, 'layers') else base_model.model.layers
        
        for layer in layers:
            if hasattr(layer, 'mlp') and hasattr(layer.mlp, 'num_experts'):
                is_moe = True
                num_experts = layer.mlp.num_experts
                num_experts_per_tok = layer.mlp.num_experts_per_tok
                sparsity_factor = num_experts_per_tok / num_experts
                break
    except:
        pass

    if is_moe:
        total_flops = attention_flops + (ffn_flops * sparsity_factor)

    return total_flops

def compute_throughput_metrics(model, tokenizer, eval_dataset, max_samples=100):
    """
    Measure throughput and latency metrics.

    Returns:
        - tokens_per_second: Throughput in tokens/sec
        - ms_per_token: Latency in ms/token
        - samples_per_second: Sample throughput
        - total_time: Total evaluation time
    """
    model.eval()

    samples = eval_dataset.select(range(min(max_samples, len(eval_dataset))))

    total_tokens = 0
    sample_times = []

    with torch.no_grad():
        for example in samples:
            prompt = example['formatted_prompt']

            inputs = tokenizer(prompt, return_tensors="pt", truncation=True,
                             max_length=MAX_LENGTH).to("cuda")
            num_tokens = inputs['input_ids'].shape[1]
            total_tokens += num_tokens

            start = time.time()
            _ = model(**inputs)
            sample_time = time.time() - start
            sample_times.append(sample_time)

    total_time = sum(sample_times)
    tokens_per_second = total_tokens / total_time if total_time > 0 else 0
    ms_per_token = (total_time / total_tokens * 1000) if total_tokens > 0 else 0
    samples_per_second = len(samples) / total_time if total_time > 0 else 0

    return {
        'tokens_per_second': tokens_per_second,
        'ms_per_token': ms_per_token,
        'samples_per_second': samples_per_second,
        'total_time': total_time,
    }

def compute_parameter_efficiency(model, num_experts_per_tok=1):
    """
    Calculate parameter efficiency metrics.

    Returns:
        - total_params: Total model parameters
        - active_params: Parameters used per forward pass
        - trainable_params: Parameters being trained
        - sparsity_ratio: Fraction of params used (active/total)
    """
    total_params = sum(p.numel() for p in model.parameters())
    trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)

    is_moe = False
    try:
        base_model = model
        if hasattr(model, 'base_model'):
            base_model = model.base_model
        if hasattr(base_model, 'model'):
            base_model = base_model.model
        
        layers = base_model.layers if hasattr(base_model, 'layers') else base_model.model.layers
        
        for layer in layers:
            if hasattr(layer, 'mlp') and hasattr(layer.mlp, 'num_experts'):
                is_moe = True
                num_experts = layer.mlp.num_experts
                break
    except:
        pass

    if is_moe:
        total_expert_params = 0
        for layer in layers:
            if hasattr(layer, 'mlp') and hasattr(layer.mlp, 'num_experts'):
                if hasattr(layer.mlp, 'gate_proj') and isinstance(layer.mlp.gate_proj, list):
                    for expert_idx in range(num_experts):
                        total_expert_params += sum(p.numel() for p in layer.mlp.gate_proj[expert_idx].parameters())
                        total_expert_params += sum(p.numel() for p in layer.mlp.up_proj[expert_idx].parameters())
                        total_expert_params += sum(p.numel() for p in layer.mlp.down_proj[expert_idx].parameters())
        
        sparsity = num_experts_per_tok / num_experts
        active_expert_params = int(total_expert_params * sparsity)
        
        non_expert_params = total_params - total_expert_params
        
        active_params = non_expert_params + active_expert_params
    else:
        active_params = total_params

    sparsity_ratio = active_params / total_params if total_params > 0 else 1.0

    return {
        'total_params': total_params,
        'active_params': active_params,
        'trainable_params': trainable_params,
        'sparsity_ratio': sparsity_ratio,
    }

def compute_memory_metrics(model):
    """
    Compute memory usage statistics.

    Returns:
        - model_size_mb: Model size in memory
        - gpu_memory_allocated_gb: GPU memory allocated
        - gpu_memory_reserved_gb: GPU memory reserved
    """
    param_size = sum(p.nelement() * p.element_size() for p in model.parameters())
    buffer_size = sum(b.nelement() * b.element_size() for b in model.buffers())
    model_size_mb = (param_size + buffer_size) / 1024 / 1024

    if torch.cuda.is_available():
        gpu_memory_allocated_gb = torch.cuda.memory_allocated() / 1024 / 1024 / 1024
        gpu_memory_reserved_gb = torch.cuda.memory_reserved() / 1024 / 1024 / 1024
    else:
        gpu_memory_allocated_gb = 0
        gpu_memory_reserved_gb = 0

    return {
        'model_size_mb': model_size_mb,
        'gpu_memory_allocated_gb': gpu_memory_allocated_gb,
        'gpu_memory_reserved_gb': gpu_memory_reserved_gb,
    }

print("evaluation functions defined")

evaluation functions defined


In [38]:
from transformers import TrainerCallback
from torch.utils.data import DataLoader
import gc
import wandb

# used for evaluations during the training loops

class ComprehensiveEvalCallback(TrainerCallback):
    

    def __init__(self, eval_dataset_for_accuracy, eval_tokenized_for_loss,
                 tokenizer, answer_tokens, eval_steps=50,
                 accuracy_samples=1000, device="cuda"):
        super().__init__()
        self.eval_dataset_for_accuracy = eval_dataset_for_accuracy
        self.eval_tokenized_for_loss = eval_tokenized_for_loss
        self.tokenizer = tokenizer
        self.answer_tokens = answer_tokens
        self.eval_steps = eval_steps
        self.accuracy_samples = accuracy_samples
        self.device = device

        self.metrics_history = []
        self.start_time = None

    def on_train_begin(self, args, state, control, **kwargs):
        self.start_time = time.time()

        print(f"Evaluating every {self.eval_steps} steps")

        return control

    def on_step_end(self, args, state, control, model=None, **kwargs):
        if state.global_step % self.eval_steps == 0 and state.global_step > 0:
            self._evaluate_and_log(args, state, model)
        return control

    def on_epoch_end(self, args, state, control, model=None, **kwargs):

        print(f"END OF EPOCH {state.epoch:.0f}")

        self._evaluate_and_log(args, state, model, is_epoch_end=True)
        return control

    def _evaluate_and_log(self, args, state, model, is_epoch_end=False):
        """Compute and log all metrics."""

        # Get training loss from state
        train_loss = None
        if len(state.log_history) > 0:
            for log in reversed(state.log_history):
                if 'loss' in log:
                    train_loss = log['loss']
                    break

        # Get learning rate
        learning_rate = None
        if len(state.log_history) > 0:
            for log in reversed(state.log_history):
                if 'learning_rate' in log:
                    learning_rate = log['learning_rate']
                    break

        # Compute evaluation loss and throughput
        eval_loss, eval_throughput, eval_latency = self._compute_eval_loss(model)

        # Compute perplexity
        perplexity = torch.exp(torch.tensor(eval_loss)).item()

        # Comprehensive MMLU evaluation
        mmlu_results = evaluate_mmlu_comprehensive(
            model=model,
            tokenizer=self.tokenizer,
            eval_dataset=self.eval_dataset_for_accuracy,
            answer_tokens=self.answer_tokens,
            device=self.device,
            max_samples=self.accuracy_samples,
            show_progress=False
        )

        # GPU memory
        peak_gpu_memory = self._get_peak_gpu_memory()

        # Calculate elapsed time
        elapsed = time.time() - self.start_time

        # Gradient norm (if available)
        grad_norm = None
        if len(state.log_history) > 0:
            for log in reversed(state.log_history):
                if 'grad_norm' in log:
                    grad_norm = log['grad_norm']
                    break

        # Store metrics
        metrics = {
            'step': state.global_step,
            'epoch': state.epoch,
            'train_loss': train_loss,
            'eval_loss': eval_loss,
            'perplexity': perplexity,
            'learning_rate': learning_rate,
            'grad_norm': grad_norm,
            # MMLU metrics
            'mmlu_accuracy': mmlu_results['accuracy'],
            'mmlu_top2_accuracy': mmlu_results['top2_accuracy'],
            'mmlu_ece': mmlu_results['ece'],
            'mmlu_correct': mmlu_results['correct'],
            'mmlu_total': mmlu_results['total'],
            # Performance metrics
            'throughput': mmlu_results['throughput'],
            'avg_latency': mmlu_results['avg_latency'],
            'eval_throughput': eval_throughput,
            'eval_latency': eval_latency,
            'peak_gpu_memory_gb': peak_gpu_memory,
            'elapsed_time': elapsed,
        }
        self.metrics_history.append(metrics)

        # Log to WandB and TensorBoard
        try:
            if wandb.run is not None:
                wandb.log({
                    'eval/step': state.global_step,
                    'eval/train_loss': train_loss if train_loss else 0,
                    'eval/eval_loss': eval_loss,
                    'eval/perplexity': perplexity,
                    'eval/mmlu_accuracy': mmlu_results['accuracy'],
                    'eval/mmlu_top2_accuracy': mmlu_results['top2_accuracy'],
                    'eval/mmlu_ece': mmlu_results['ece'],
                    'eval/throughput': mmlu_results['throughput'],
                    'eval/avg_latency': mmlu_results['avg_latency'],
                    'eval/peak_gpu_memory_gb': peak_gpu_memory,
                })
        except:
            pass

        # Print metrics
        step_info = f"Epoch {state.epoch:.2f}" if is_epoch_end else f"Step {state.global_step}"

        print(f"EVALUATION at {step_info}")
        print(f"  Train Loss:      {train_loss:.4f}" if train_loss else "  Train Loss:      N/A")
        print(f"  Eval Loss:       {eval_loss:.4f}")
        print(f"  MMLU Accuracy:   {mmlu_results['accuracy']:.4f} ({mmlu_results['correct']}/{mmlu_results['total']})")
        print(f"  Perplexity:      {perplexity:.2f}")
        print(f"  Throughput:      {mmlu_results['throughput']:.2f} samples/sec")
        print(f"  Avg Latency:     {mmlu_results['avg_latency']:.4f} sec/sample")
        print(f"  Peak GPU Memory: {peak_gpu_memory:.2f} GB")
        print(f"  ECE (Calibration): {mmlu_results['ece']:.4f}")
        print(f"  Top-2 Accuracy:  {mmlu_results['top2_accuracy']:.4f}")
        if learning_rate:
            print(f"  Learning Rate:   {learning_rate:.2e}")
        if grad_norm:
            print(f"  Gradient Norm:   {grad_norm:.4f}")

        print(f" Elapsed Time:    {elapsed/60:.1f} min")


        # Clean up
        torch.cuda.empty_cache()
        gc.collect()
        model.train()

    def _compute_eval_loss(self, model, num_batches=50):
        """Compute average loss on evaluation set."""
        model.eval()
        total_loss = 0
        num_samples = 0

        eval_dataloader = DataLoader(
            self.eval_tokenized_for_loss,
            batch_size=1,
            shuffle=False
        )

        start_time = time.time()

        with torch.no_grad():
            for i, batch in enumerate(eval_dataloader):
                if i >= num_batches:
                    break

                processed_batch = {}
                for k, v in batch.items():
                    if isinstance(v, list):
                        v = torch.tensor(v)
                    # Ensure tensor is 2D (add batch dimension if needed)
                    if v.dim() == 1:
                        v = v.unsqueeze(0)
                    processed_batch[k] = v.to(self.device)
                
                if 'labels' not in processed_batch and 'input_ids' in processed_batch:
                    processed_batch['labels'] = processed_batch['input_ids'].clone()
                
                outputs = model(**processed_batch)
                
                # Skip if loss is still None (shouldn't happen now)
                if outputs.loss is None:
                    continue
                    
                total_loss += outputs.loss.item()
                num_samples += 1

        end_time = time.time()
        duration = end_time - start_time

        avg_loss = total_loss / num_samples if num_samples > 0 else 0
        throughput = num_samples / duration if duration > 0 else 0.0
        latency = duration / num_samples if num_samples > 0 else 0.0

        return avg_loss, throughput, latency

    def _get_peak_gpu_memory(self):
        """Get peak GPU memory and reset stats."""
        if torch.cuda.is_available():
            max_mem = torch.cuda.max_memory_allocated() / (1024**3)
            torch.cuda.reset_peak_memory_stats()
            return max_mem
        return 0.0

    def on_train_end(self, args, state, control, **kwargs):
        elapsed = time.time() - self.start_time

        print(f"TRAINING COMPLETED in {elapsed/60:.1f} minutes")

        # Print summary table
        if self.metrics_history:
            print("TRAINING METRICS SUMMARY:")
            print(f"{'Step':<8} {'Epoch':<7} {'Train':<8} {'Eval':<8} {'Perp':<7} {'MMLU':<8} {'Top-2':<8} {'ECE':<7} {'Latency':<9}")
            print(f"{'':8} {'':7} {'Loss':<8} {'Loss':<8} {'':7} {'Acc':<8} {'Acc':<8} {'':7} {'(sec)':<9}")
            print("-" * 80)
            for m in self.metrics_history:
                train_loss_str = f"{m['train_loss']:.4f}" if m['train_loss'] else "N/A"
                print(f"{m['step']:<8} {m['epoch']:<7.2f} {train_loss_str:<8} {m['eval_loss']:<8.4f} "
                      f"{m['perplexity']:<7.2f} {m['mmlu_accuracy']:<8.4f} {m['mmlu_top2_accuracy']:<8.4f} "
                      f"{m['mmlu_ece']:<7.4f} {m['avg_latency']:<9.4f}")

        return control


print(" Comprehensive evaluation callback with batched eval defined")

 Comprehensive evaluation callback with batched eval defined


## Standardized Comprehensive Evaluation Function

In [14]:
import json
import os
import numpy as np

def evaluate_model_comprehensive(
    model, 
    tokenizer, 
    eval_dataset, 
    answer_tokens, 
    model_name="model",
    device="cuda",
    max_samples_mmlu=1000,
    max_samples_throughput=100,
    show_progress=True
):
    """
    Standardized comprehensive evaluation function for all models.
    
    Collects MMLU accuracy, FLOPs, throughput, parameter efficiency, and memory metrics.
    Provides consistent verbose output format and saves results to JSON.
    
    Args:
        model: Model to evaluate
        tokenizer: Tokenizer
        eval_dataset: Evaluation dataset
        answer_tokens: Dictionary with answer token IDs
        model_name: Name for saving results
        device: Device to run on
        max_samples_mmlu: Number of samples for MMLU evaluation
        max_samples_throughput: Number of samples for throughput measurement
        show_progress: Whether to show progress bar
    
    Returns:
        Dictionary with all comprehensive metrics
    """
    print(f"\n{'='*80}")
    print(f"COMPREHENSIVE EVALUATION: {model_name}")
    print(f"{'='*80}\n")
    
    print(f"Running MMLU evaluation (n={max_samples_mmlu})...")
    mmlu_results = evaluate_mmlu_comprehensive(
        model=model,
        tokenizer=tokenizer,
        eval_dataset=eval_dataset,
        answer_tokens=answer_tokens,
        device=device,
        max_samples=max_samples_mmlu,
        show_progress=show_progress
    )
    
    print(f"\nComputing FLOPs...")
    flops = compute_model_flops(model, seq_length=MAX_LENGTH)
    
    print(f"Measuring throughput (n={max_samples_throughput})...")
    throughput_metrics = compute_throughput_metrics(
        model=model,
        tokenizer=tokenizer,
        eval_dataset=eval_dataset,
        max_samples=max_samples_throughput
    )
    
    print(f"Analyzing parameter efficiency...")
    num_experts_per_tok = 1
    try:
        base_model = model
        if hasattr(model, 'base_model'):
            base_model = model.base_model
        if hasattr(base_model, 'model'):
            base_model = base_model.model
        layers = base_model.layers if hasattr(base_model, 'layers') else base_model.model.layers
        for layer in layers:
            if hasattr(layer, 'mlp') and hasattr(layer.mlp, 'num_experts'):
                num_experts_per_tok = layer.mlp.num_experts_per_tok
                break
    except:
        pass
    
    param_metrics = compute_parameter_efficiency(
        model=model,
        num_experts_per_tok=num_experts_per_tok
    )
    
    print(f"Collecting memory metrics...")
    memory_metrics = compute_memory_metrics(model)
    
    results = {
        'model_name': model_name,
        'accuracy': mmlu_results['accuracy'],
        'top2_accuracy': mmlu_results['top2_accuracy'],
        'ece': mmlu_results['ece'],
        'flops': flops,
        'tokens_per_second': throughput_metrics['tokens_per_second'],
        'ms_per_token': throughput_metrics['ms_per_token'],
        'samples_per_second': throughput_metrics['samples_per_second'],
        'total_params': param_metrics['total_params'],
        'active_params': param_metrics['active_params'],
        'trainable_params': param_metrics['trainable_params'],
        'sparsity_ratio': param_metrics['sparsity_ratio'],
        'model_size_mb': memory_metrics['model_size_mb'],
        'gpu_memory_allocated_gb': memory_metrics['gpu_memory_allocated_gb'],
        'gpu_memory_reserved_gb': memory_metrics['gpu_memory_reserved_gb'],
    }
    
    print(f"\n{'='*80}")
    print(f"COMPREHENSIVE METRICS: {model_name}")
    print(f"{'='*80}")
    print(f"Accuracy Metrics:")
    print(f"  MMLU Accuracy: {results['accuracy']:.4f}")
    print(f"  Top-2 Accuracy: {results['top2_accuracy']:.4f}")
    print(f"  ECE: {results['ece']:.4f}")
    print(f"\n Computational Efficiency:")
    print(f"  FLOPs per forward pass: {results['flops']/1e9:.2f}G")
    print(f"  Tokens/second: {results['tokens_per_second']:.2f}")
    print(f"  ms/token: {results['ms_per_token']:.2f}")
    print(f"  Samples/second: {results['samples_per_second']:.2f}")
    print(f"\n Parameter Efficiency:")
    print(f"  Total parameters: {results['total_params']/1e9:.2f}B")
    print(f"  Active parameters: {results['active_params']/1e9:.2f}B")
    print(f"  Trainable parameters: {results['trainable_params']/1e6:.2f}M")
    print(f"  Sparsity ratio: {results['sparsity_ratio']:.2%}")
    print(f"\n Memory Usage:")
    print(f"  Model size: {results['model_size_mb']:.2f} MB")
    print(f"  GPU allocated: {results['gpu_memory_allocated_gb']:.2f} GB")
    print(f"  GPU reserved: {results['gpu_memory_reserved_gb']:.2f} GB")
    print(f"{'='*80}\n")
    
    os.makedirs("results", exist_ok=True)
    results_file = f"results/{model_name}_comprehensive.json"
    with open(results_file, 'w') as f:
        json.dump({k: v for k, v in results.items()
                   if not isinstance(v, np.ndarray)}, f, indent=2, default=str)
    print(f"Results saved to: {results_file}\n")
    
    return results

## Dense Model Baseline Evaluation

In [15]:
import json
import os
import gc
import numpy as np
import wandb

baseline_comprehensive = evaluate_model_comprehensive(
    model=base_model,
    tokenizer=tokenizer,
    eval_dataset=eval_dataset,
    answer_tokens=ANSWER_TOKENS,
    model_name="baseline",
    device="cuda",
    max_samples_mmlu=1000,
    max_samples_throughput=100,
    show_progress=True
)

try:
    if wandb.run is not None:
        wandb.log({
            'baseline/accuracy': baseline_comprehensive['accuracy'],
            'baseline/flops_billions': baseline_comprehensive['flops']/1e9,
            'baseline/tokens_per_second': baseline_comprehensive['tokens_per_second'],
            'baseline/ms_per_token': baseline_comprehensive['ms_per_token'],
            'baseline/active_params_billions': baseline_comprehensive['active_params']/1e9,
            'baseline/gpu_memory_gb': baseline_comprehensive['gpu_memory_allocated_gb'],
        })
except:
    pass

baseline_accuracy = baseline_comprehensive['accuracy']


COMPREHENSIVE EVALUATION: baseline

Running MMLU evaluation (n=1000)...


Evaluating: 100%|██████████| 1000/1000 [01:30<00:00, 11.04it/s]



Computing FLOPs...
Measuring throughput (n=100)...
Analyzing parameter efficiency...

COMPREHENSIVE METRICS: baseline
Accuracy Metrics:
  MMLU Accuracy: 0.6590
  Top-2 Accuracy: 0.8370
  ECE: 0.0796

 Computational Efficiency:
  FLOPs per forward pass: 8796.09G
  Tokens/second: 3116.55
  ms/token: 0.32
  Samples/second: 10.96

 Parameter Efficiency:
  Total parameters: 7.24B
  Active parameters: 7.24B
  Trainable parameters: 262.41M
  Sparsity ratio: 100.00%

 Memory Usage:
  Model size: 7156.51 MB
  GPU allocated: 7.03 GB
  GPU reserved: 7.86 GB

Results saved to: results/baseline_comprehensive.json



## Knowledge distillation setup

In [16]:
teacher_model = base_model.eval()
teacher_baseline_results = baseline_comprehensive.copy()
print(f"Teacher model set (dense). Teacher accuracy: {teacher_baseline_results['accuracy']:.4f}")

KD_CONFIG_STANDARD = {
    'kd_alpha': 0.5,
    'temperature': 4.0,
    'routing_kd_weight': 0.0,
    'expert_spec_weight': 0.0,
    'enable_routing_kd': False,
    'enable_ka': False,
    'enable_sar': False,
    'enable_non_activated': False,
    'router_aux_loss_coef': 0.001,
    'name': 'Standard KD',
}

KD_CONFIG_ROUTER_STABLE = {
    'kd_alpha': 0.6,
    'temperature': 5.0,
    'routing_kd_weight': 0.1,
    'expert_spec_weight': 0.0,
    'enable_routing_kd': True,
    'enable_ka': False,
    'enable_sar': False,
    'enable_non_activated': False,
    'router_aux_loss_coef': 0.01,
    'name': 'Router-Stable KD',
}

print("KD configs ready (Standard, Router-Stable)")

Teacher model set (dense). Teacher accuracy: 0.6590
KD configs ready (Standard, Router-Stable)


## Dense Model Knowledge Distillation Trainer

In [18]:
from transformers import Trainer
import torch.nn.functional as F
import torch

class DenseKDTrainer(Trainer):
    """
    Knowledge Distillation Trainer for Dense Models.
    Loss: L_total = (1-α)*L_NTP + α*L_KD
    
    Where:
    - L_NTP: Next Token Prediction loss (standard supervised learning)
    - L_KD: Knowledge Distillation loss (KL divergence between teacher and student logits)
    - α: Weighting factor for KD loss (typically 0.5)
    """
    def __init__(self, teacher_model=None, kd_config=None, *args, **kwargs):
        super().__init__(*args, **kwargs)
        self.teacher_model = teacher_model
        self.kd_config = kd_config or {}
        self.kd_alpha = self.kd_config.get('kd_alpha', 0.5)
        self.temperature = self.kd_config.get('temperature', 4.0)
        
        if self.teacher_model is not None:
            self.teacher_model.eval()
            for param in self.teacher_model.parameters():
                param.requires_grad = False
            print(f"Teacher model loaded (frozen, KD alpha={self.kd_alpha}, T={self.temperature})")

    def compute_loss(self, model, inputs, return_outputs=False, num_items_in_batch=None):
        """
        Compute loss with Knowledge Distillation.
        
        Args:
            model: The student model
            inputs: Input dictionary with input_ids, attention_mask, labels, etc.
            return_outputs: Whether to return model outputs along with loss
            num_items_in_batch: Number of items in batch (for newer transformers versions, optional)
        """
        outputs = model(**inputs)
        ntp_loss = outputs.loss if hasattr(outputs, 'loss') else outputs[0]
        device = ntp_loss.device if hasattr(ntp_loss, 'device') else next(model.parameters()).device

        kd_loss = torch.tensor(0.0, device=device)

        if self.teacher_model is not None and self.kd_alpha > 0:
            tdev = next(self.teacher_model.parameters()).device
            tinputs = {k: (v.to(tdev) if hasattr(v, 'to') else v) for k, v in inputs.items()}
            
            with torch.no_grad():
                t_out = self.teacher_model(**tinputs)

            s_logits = outputs.logits / self.temperature
            t_logits = t_out.logits / self.temperature

            kd_loss = F.kl_div(
                F.log_softmax(s_logits, dim=-1),
                F.softmax(t_logits, dim=-1),
                reduction='batchmean'
            ) * (self.temperature ** 2)

        total_loss = (1 - self.kd_alpha) * ntp_loss + self.kd_alpha * kd_loss

        return (total_loss, outputs) if return_outputs else total_loss

## Dense Model Knowledge Distillation Training Setup

In [19]:
from transformers import TrainingArguments, DataCollatorForLanguageModeling, AutoModelForCausalLM, BitsAndBytesConfig
from peft import get_peft_model, LoraConfig, TaskType, prepare_model_for_kbit_training
import os
import torch
import time

USE_KNOWLEDGE_DISTILLATION_DENSE = True

USE_SUBSET_DENSE = True
SUBSET_PERCENTAGE_DENSE = 0.2

TRAINING_CONFIG_DENSE = {
    "learning_rate": 2e-4,
    "per_device_train_batch_size": 4,
    "per_device_eval_batch_size": 4,
    "gradient_accumulation_steps": 4,
    "warmup_ratio": 0.1,
    "num_train_epochs": 1,
    "logging_steps": 25,
    "save_steps": 100,
    "max_steps": 250,
    "fp16": True,
    "bf16": False,
    "save_total_limit": 2,
}

MAX_LENGTH_DENSE = 512

LORA_CONFIG_DENSE = {
    "r": 16,
    "lora_alpha": 32,
    "lora_dropout": 0.05,
    "target_modules": ["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
    "task_type": TaskType.CAUSAL_LM,
}

KD_CONFIG_DENSE = {
    'kd_alpha': 0.5,
    'temperature': 4.0,
    'name': 'Standard KD',
}

OUTPUT_DIR_DENSE = "./dense_model_kd" if USE_KNOWLEDGE_DISTILLATION_DENSE else "./dense_model_standard"

print("=" * 80)
print("DENSE MODEL TRAINING CONFIGURATION")
print("=" * 80)
print(f"Training Mode: {'Knowledge Distillation' if USE_KNOWLEDGE_DISTILLATION_DENSE else 'Standard'}")
print(f"KD Alpha: {KD_CONFIG_DENSE['kd_alpha']}")
print(f"Temperature: {KD_CONFIG_DENSE['temperature']}")
print(f"Output Directory: {OUTPUT_DIR_DENSE}")
print(f"Max Steps: {TRAINING_CONFIG_DENSE['max_steps']}")
print("=" * 80)

if 'teacher_model' not in globals() or teacher_model is None:
    if 'base_model' in globals():
        teacher_model = base_model.eval()
        for param in teacher_model.parameters():
            param.requires_grad = False
        print("\nTeacher model set from base_model")
    else:
        raise ValueError("teacher_model or base_model must be available!")

print("\nSetting up student model...")

bnb_config_dense = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True,
)

if 'base_model' in globals() and hasattr(base_model, 'config'):
    MODEL_NAME_DENSE = base_model.config._name_or_path if hasattr(base_model.config, '_name_or_path') else "mistralai/Mistral-7B-v0.1"
else:
    MODEL_NAME_DENSE = "mistralai/Mistral-7B-v0.1"

print(f"Loading student model from: {MODEL_NAME_DENSE}")

student_model_dense = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME_DENSE,
    quantization_config=bnb_config_dense,
    device_map="auto",
    trust_remote_code=True,
)

student_model_dense.config.use_cache = False

student_model_dense = prepare_model_for_kbit_training(student_model_dense)
print("Student model prepared for k-bit training")

peft_config_dense = LoraConfig(**LORA_CONFIG_DENSE)
student_model_dense = get_peft_model(student_model_dense, peft_config_dense)
print("LoRA adapters applied")

student_model_dense.gradient_checkpointing_enable()
student_model_dense.train()
print("Student model set to training mode")

trainable_params = sum(p.numel() for p in student_model_dense.parameters() if p.requires_grad)
total_params = sum(p.numel() for p in student_model_dense.parameters())
print(f"Trainable parameters: {trainable_params/1e6:.2f}M / {total_params/1e9:.2f}B ({100*trainable_params/total_params:.2f}%)")

if 'tokenized_train' not in globals() or 'tokenized_eval' not in globals():
    if 'dataset' not in globals():
        raise ValueError(
            "Missing prerequisites!\n"
            "Please run these cells in order:\n"
            "1. Dataset loading cell (creates 'dataset')\n"
            "2. Dataset tokenization cell (creates 'tokenized_train' and 'tokenized_eval')\n"
            "3. Then run this Dense KD training cell"
        )
    
    print("tokenized_train/tokenized_eval not found. Creating them from 'dataset'...")
    print("   (This should normally be done in the tokenization cell)")
    
    if 'tokenizer' not in globals():
        raise ValueError("tokenizer must be defined! Run model loading cells first.")
    
    if 'MAX_LENGTH_DENSE' in globals():
        max_length = MAX_LENGTH_DENSE
    elif 'TRAINING_CONFIG_DENSE' in globals() and 'max_length' in TRAINING_CONFIG_DENSE:
        max_length = TRAINING_CONFIG_DENSE['max_length']
    elif 'TRAINING_CONFIG' in globals() and 'max_length' in TRAINING_CONFIG:
        max_length = TRAINING_CONFIG['max_length']
    else:
        max_length = 512
        print("   Warning: Using default max_length=512")
    
    def tokenize_function_for_training(examples):
        """Tokenize WITH answers for training."""
        return tokenizer(
            examples['formatted_prompt_with_answer'],
            truncation=True,
            max_length=max_length,
            padding=False,
        )
    
    def tokenize_function_for_eval(examples):
        """Tokenize WITHOUT answers for evaluation."""
        return tokenizer(
            examples['formatted_prompt'],
            truncation=True,
            max_length=max_length,
            padding=False,
        )
    
    print("   Tokenizing training data (with answers)...")
    tokenized_train = dataset['train'].map(
        tokenize_function_for_training,
        batched=True,
        remove_columns=dataset['train'].column_names,
        desc="Tokenizing training data"
    )
    
    print("   Tokenizing eval data (without answers)...")
    tokenized_eval = dataset['test'].map(
        tokenize_function_for_eval,
        batched=True,
        remove_columns=dataset['test'].column_names,
        desc="Tokenizing eval data"
    )
    
    print(f"Created tokenized datasets: {len(tokenized_train):,} train, {len(tokenized_eval):,} eval")

if 'train_dataset' in globals() and train_dataset is not None:
    base_train_dataset = train_dataset
    print("Using existing train_dataset")
else:
    base_train_dataset = tokenized_train
    print("Using tokenized_train as base dataset")

if 'val_dataset' in globals() and val_dataset is not None:
    base_val_dataset = val_dataset
    print("Using existing val_dataset")
else:
    base_val_dataset = tokenized_eval
    print("Using tokenized_eval as base dataset")

if USE_SUBSET_DENSE:
    subset_size = int(len(base_train_dataset) * SUBSET_PERCENTAGE_DENSE)
    train_dataset_subset_dense = base_train_dataset.select(range(subset_size))
    print(f"Using {SUBSET_PERCENTAGE_DENSE*100}% of training data: {len(train_dataset_subset_dense)} samples")
else:
    train_dataset_subset_dense = base_train_dataset

if 'eval_dataset' in globals() and eval_dataset is not None:
    eval_dataset_dense = eval_dataset
elif 'val_dataset' in globals() and val_dataset is not None:
    eval_dataset_dense = val_dataset
else:
    eval_dataset_dense = base_val_dataset
    print("Using tokenized_eval for evaluation")

training_args_dense = TrainingArguments(
    output_dir=OUTPUT_DIR_DENSE,
    **TRAINING_CONFIG_DENSE,
    save_strategy="steps",
    load_best_model_at_end=False,
    report_to="wandb" if 'wandb' in globals() else None,
    run_name=f"dense_kd_{KD_CONFIG_DENSE['kd_alpha']}" if USE_KNOWLEDGE_DISTILLATION_DENSE else "dense_standard",
    logging_first_step=True,
    logging_strategy="steps",
    eval_strategy="no",
)

data_collator_dense = DataCollatorForLanguageModeling(tokenizer=tokenizer, mlm=False)

print("\nInitializing trainer...")

if USE_KNOWLEDGE_DISTILLATION_DENSE:
    print("  Using DenseKDTrainer for knowledge distillation")
    print("  No evaluation callback - evaluation will be done post-training only")
    trainer_dense = DenseKDTrainer(
        model=student_model_dense,
        args=training_args_dense,
        train_dataset=train_dataset_subset_dense,
        eval_dataset=eval_dataset_dense,
        data_collator=data_collator_dense,
        teacher_model=teacher_model,
        kd_config=KD_CONFIG_DENSE,
    )
else:
    print("  Using standard Trainer")
    print("  No evaluation callback - evaluation will be done post-training only")
    from transformers import Trainer
    trainer_dense = Trainer(
        model=student_model_dense,
        args=training_args_dense,
        train_dataset=train_dataset_subset_dense,
        eval_dataset=eval_dataset_dense,
        data_collator=data_collator_dense,
    )

print("Trainer initialized successfully!")

DENSE MODEL TRAINING CONFIGURATION
Training Mode: Knowledge Distillation
KD Alpha: 0.5
Temperature: 4.0
Output Directory: ./dense_model_kd
Max Steps: 250

Setting up student model...
Loading student model from: mistralai/Mistral-7B-v0.1


Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

Student model prepared for k-bit training
LoRA adapters applied
Student model set to training mode
Trainable parameters: 41.94M / 3.79B (1.11%)
tokenized_train/tokenized_eval not found. Creating them from 'dataset'...
   (This should normally be done in the tokenization cell)
   Tokenizing training data (with answers)...


Tokenizing training data:   0%|          | 0/68940 [00:00<?, ? examples/s]

   Tokenizing eval data (without answers)...


Tokenizing eval data:   0%|          | 0/29547 [00:00<?, ? examples/s]

Created tokenized datasets: 68,940 train, 29,547 eval
Using tokenized_train as base dataset
Using tokenized_eval as base dataset
Using 20.0% of training data: 13788 samples

Initializing trainer...
  Using DenseKDTrainer for knowledge distillation
  No evaluation callback - evaluation will be done post-training only
Teacher model loaded (frozen, KD alpha=0.5, T=4.0)
Trainer initialized successfully!


## Execute Dense Model Knowledge Distillation Training

In [20]:
import torch
import gc

print("\n" + "=" * 80)
print("STARTING DENSE MODEL TRAINING")
print("=" * 80)
print("Note: No evaluation during training - will evaluate after training completes")

torch.cuda.empty_cache()
gc.collect()

train_result_dense = trainer_dense.train()

print("\n" + "=" * 80)
print("DENSE MODEL TRAINING COMPLETE")
print("=" * 80)

print("\nFinal training metrics:")
for key, value in train_result_dense.metrics.items():
    if isinstance(value, float):
        print(f"  {key}: {value:.4f}")
    else:
        print(f"  {key}: {value}")

print(f"\nSaving model to {OUTPUT_DIR_DENSE}/final_model...")
trainer_dense.save_model(f"{OUTPUT_DIR_DENSE}/final_model")
tokenizer.save_pretrained(f"{OUTPUT_DIR_DENSE}/final_model")


STARTING DENSE MODEL TRAINING
Note: No evaluation during training - will evaluate after training completes


Step,Training Loss
1,72.864900
25,51.160300
50,39.173000
75,38.267600
100,40.752800
125,39.655800
150,36.559200
175,37.060600
200,35.782100
225,34.395400



DENSE MODEL TRAINING COMPLETE

Final training metrics:
  train_runtime: 652.0067
  train_samples_per_second: 6.1350
  train_steps_per_second: 0.3830
  total_flos: 85174800326590464.0000
  train_loss: 38.9325
  epoch: 0.2901

Saving model to ./dense_model_kd/final_model...


('./dense_model_kd/final_model/tokenizer_config.json',
 './dense_model_kd/final_model/special_tokens_map.json',
 './dense_model_kd/final_model/tokenizer.json')

## Dense Knowledge Distillation Model Evaluation

In [21]:
import json
import os
import numpy as np
import wandb

student_model_dense.eval()

kd_dense_comprehensive = evaluate_model_comprehensive(
    model=student_model_dense,
    tokenizer=tokenizer,
    eval_dataset=eval_dataset,
    answer_tokens=ANSWER_TOKENS,
    model_name="dense_kd",
    device="cuda",
    max_samples_mmlu=1000,
    max_samples_throughput=100,
    show_progress=True
)

if 'baseline_comprehensive' in globals():
    print("\n" + "=" * 80)
    print("COMPARISON WITH BASELINE DENSE MODEL")
    print("=" * 80)
    
    acc_diff = kd_dense_comprehensive['accuracy'] - baseline_comprehensive['accuracy']
    acc_change = (acc_diff / baseline_comprehensive['accuracy']) * 100
    
    print("\nAccuracy Comparison:")
    print(f"  Baseline Accuracy:     {baseline_comprehensive['accuracy']:.4f}")
    print(f"  KD Dense Accuracy:     {kd_dense_comprehensive['accuracy']:.4f}")
    print(f"  Difference:            {acc_diff:+.4f} ({acc_change:+.2f}%)")
    
    print("\nEfficiency Comparison:")
    print(f"  Baseline Tokens/sec:   {baseline_comprehensive['tokens_per_second']:.2f}")
    print(f"  KD Dense Tokens/sec:    {kd_dense_comprehensive['tokens_per_second']:.2f}")
    diff_tokens = kd_dense_comprehensive['tokens_per_second'] - baseline_comprehensive['tokens_per_second']
    print(f"  Difference:            {diff_tokens:+.2f} ({100*diff_tokens/baseline_comprehensive['tokens_per_second']:+.2f}%)")
    
    print("\nMemory Comparison:")
    print(f"  Baseline GPU Memory:   {baseline_comprehensive['gpu_memory_allocated_gb']:.2f} GB")
    print(f"  KD Dense GPU Memory:   {kd_dense_comprehensive['gpu_memory_allocated_gb']:.2f} GB")
    diff_mem = kd_dense_comprehensive['gpu_memory_allocated_gb'] - baseline_comprehensive['gpu_memory_allocated_gb']
    print(f"  Difference:            {diff_mem:+.2f} GB")

try:
    if wandb.run is not None:
        wandb.log({
            'dense_kd/accuracy': kd_dense_comprehensive['accuracy'],
            'dense_kd/top2_accuracy': kd_dense_comprehensive['top2_accuracy'],
            'dense_kd/ece': kd_dense_comprehensive['ece'],
            'dense_kd/flops_billions': kd_dense_comprehensive['flops']/1e9,
            'dense_kd/tokens_per_second': kd_dense_comprehensive['tokens_per_second'],
            'dense_kd/ms_per_token': kd_dense_comprehensive['ms_per_token'],
            'dense_kd/active_params_billions': kd_dense_comprehensive['active_params']/1e9,
            'dense_kd/gpu_memory_gb': kd_dense_comprehensive['gpu_memory_allocated_gb'],
        })
except:
    pass

print("\nPost-training evaluation complete!")
print("Results saved to results/dense_kd_comprehensive.json")


COMPREHENSIVE EVALUATION: dense_kd

Running MMLU evaluation (n=1000)...


Evaluating: 100%|██████████| 1000/1000 [00:41<00:00, 24.03it/s]



Computing FLOPs...
Measuring throughput (n=100)...
Analyzing parameter efficiency...

COMPREHENSIVE METRICS: dense_kd
Accuracy Metrics:
  MMLU Accuracy: 0.6540
  Top-2 Accuracy: 0.8250
  ECE: 0.0663

 Computational Efficiency:
  FLOPs per forward pass: 8796.09G
  Tokens/second: 7537.80
  ms/token: 0.13
  Samples/second: 26.50

 Parameter Efficiency:
  Total parameters: 3.79B
  Active parameters: 3.79B
  Trainable parameters: 41.94M
  Sparsity ratio: 100.00%

 Memory Usage:
  Model size: 4489.02 MB
  GPU allocated: 11.86 GB
  GPU reserved: 15.74 GB

Results saved to: results/dense_kd_comprehensive.json


COMPARISON WITH BASELINE DENSE MODEL

Accuracy Comparison:
  Baseline Accuracy:     0.6590
  KD Dense Accuracy:     0.6540
  Difference:            -0.0050 (-0.76%)

Efficiency Comparison:
  Baseline Tokens/sec:   3097.43
  KD Dense Tokens/sec:    7537.80
  Difference:            +4440.37 (+143.36%)

Memory Comparison:
  Baseline GPU Memory:   7.03 GB
  KD Dense GPU Memory:   11.86 GB


## Baseline MoE implementation

In [17]:
import torch
import torch.nn as nn

class MoELayer(nn.Module):
    """
    MoE Layer with router initialization for top-2 routing and vectorized output combination.
    """
    
    def __init__(self, hidden_size, intermediate_size, num_experts=8,
                 num_experts_per_tok=2, router_jitter_noise=0.0,
                 router_aux_loss_coef=0.001, bnb_config=None, device="cuda",
                 init_on_cpu=True, dtype=torch.bfloat16, enable_cpu_offload=True):
        super().__init__()
        
        self.hidden_size = hidden_size
        self.intermediate_size = intermediate_size
        self.num_experts = num_experts
        self.num_experts_per_tok = num_experts_per_tok
        self.router_jitter_noise = router_jitter_noise
        self.router_aux_loss_coef = router_aux_loss_coef
        self.bnb_config = bnb_config
        self.device = device
        self.dtype = dtype
        self.enable_cpu_offload = False
        
        LinearClass = nn.Linear
        self.compute_dtype = dtype
        init_device = 'cpu' if init_on_cpu else device
        linear_kwargs = {'device': init_device, 'dtype': dtype}
        
        self.gate_proj = nn.ModuleList([
            LinearClass(hidden_size, intermediate_size, bias=False, **linear_kwargs)
            for _ in range(num_experts)
        ])
        self.up_proj = nn.ModuleList([
            LinearClass(hidden_size, intermediate_size, bias=False, **linear_kwargs)
            for _ in range(num_experts)
        ])
        self.down_proj = nn.ModuleList([
            LinearClass(intermediate_size, hidden_size, bias=False, **linear_kwargs)
            for _ in range(num_experts)
        ])
        
        self.router = nn.Linear(hidden_size, num_experts, bias=True, device=device, dtype=dtype)
        with torch.no_grad():
            self.router.weight.zero_()
            self.router.bias[0] = 10.0
            self.router.bias[1] = 9.0
            if num_experts > 2:
                self.router.bias[2:] = -10.0
        
        self._last_router_probs = None
        self._collect_router_logits = False
        self._experts_on_gpu = set()
    
    def forward(self, hidden_states):
        """
        Vectorized forward pass.
        """
        original_dtype = hidden_states.dtype
        hidden_states_reshaped = hidden_states.view(-1, hidden_states.size(-1))
        
        router_input = hidden_states_reshaped.to(self.router.weight.dtype)
        router_logits = self.router(router_input)
        
        if self.training and self.router_jitter_noise > 0:
            router_logits = router_logits + torch.normal(
                0, self.router_jitter_noise, size=router_logits.shape, device=router_logits.device
            )
        
        router_probs = torch.softmax(router_logits, dim=-1)
        top_k_probs, top_k_indices = torch.topk(router_probs, self.num_experts_per_tok, dim=-1)

        top_k_probs = top_k_probs / top_k_probs.sum(dim=-1, keepdim=True)
        
        if self.training:
            self._last_router_probs = router_probs.view(hidden_states.size(0), hidden_states.size(1), -1)
        
        output = torch.zeros_like(hidden_states_reshaped)
        
        for expert_idx in range(self.num_experts):
            expert_mask = (top_k_indices == expert_idx).any(dim=-1)
            if expert_mask.sum() == 0:
                continue
            
            expert_selected_mask = (top_k_indices == expert_idx)
            expert_weights = (top_k_probs * expert_selected_mask.float()).sum(dim=-1)
            
            expert_hidden = hidden_states_reshaped[expert_mask]
            
            if self.compute_dtype is not None and expert_hidden.dtype != self.compute_dtype:
                expert_hidden = expert_hidden.to(dtype=self.compute_dtype)
            
            gate_out = torch.nn.functional.silu(self.gate_proj[expert_idx](expert_hidden))
            up_out = self.up_proj[expert_idx](expert_hidden)
            expert_out = gate_out * up_out
            expert_out = self.down_proj[expert_idx](expert_out)
            expert_out = expert_out.to(dtype=output.dtype)
            
            output[expert_mask] += expert_weights[expert_mask].unsqueeze(-1) * expert_out
        
        output = output.view_as(hidden_states)
        return output.to(original_dtype)
    
    def compute_auxiliary_loss(self):
        """
        Compute load balancing auxiliary loss.
        """
        if self._last_router_probs is None:
            return torch.tensor(0.0, device=self.router.weight.device)
        expert_freq = self._last_router_probs.mean(dim=[0, 1])
        router_confidence = self._last_router_probs.mean(dim=[0, 1])
        aux_loss = torch.sum(expert_freq * router_confidence) * self.num_experts
        return aux_loss

print("MoELayer class defined")
print("Router initialized for top-2 routing")

MoELayer class defined
Router initialized for top-2 routing


In [18]:
import gc
import psutil
import os
import ctypes
import torch

def replace_ffn_with_moe(model, num_experts=8, num_experts_per_tok=2,
                         router_jitter_noise=0.0, router_aux_loss_coef=0.001,
                         bnb_config=None, ram_threshold=50.0, use_disk_offload=True,
                         layer_indices=None, half_width=False, enable_cpu_offload=True):
    """
    Replace dense FFN layers with MoE layers.
    """
    os.environ['PYTORCH_CUDA_ALLOC_CONF'] = 'max_split_size_mb:128'
    
    config = model.config
    hidden_size = config.hidden_size
    
    if half_width:
        intermediate_size = config.intermediate_size // 2
        print(f"   Using HALF-WIDTH experts (intermediate_size // 2)")
    else:
        intermediate_size = config.intermediate_size
    
    print(f"Model configuration:")
    print(f"  Hidden size: {hidden_size}")
    print(f"  Original intermediate size: {config.intermediate_size}")
    print(f"  MoE intermediate size: {intermediate_size}")
    print(f"  Number of layers: {config.num_hidden_layers}")
    print(f"  Experts per layer: {num_experts}")
    print(f"  Experts per token: {num_experts_per_tok}")
    
    if not torch.cuda.is_available():
        raise RuntimeError("CUDA not available!")
    
    device = torch.device("cuda")
    if bnb_config and hasattr(bnb_config, 'bnb_4bit_compute_dtype'):
        compute_dtype = bnb_config.bnb_4bit_compute_dtype
    elif bnb_config and hasattr(bnb_config, 'llm_int8_threshold'):
        compute_dtype = torch.bfloat16
    else:
        compute_dtype = torch.bfloat16
    
    print(f"\n Using GPU for weight processing")
    print(f"  GPU Memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.2f} GB")
    print(f"  Compute dtype: {compute_dtype}")
    
    def check_ram():
        return psutil.virtual_memory().percent
    
    def cleanup(aggressive=False):
        gc.collect()
        torch.cuda.empty_cache()
        if aggressive:
            torch.cuda.synchronize()
        try:
            ctypes.CDLL("libc.so.6").malloc_trim(0)
        except:
            pass
        gc.collect()
    
    def print_memory_stats(label=""):
        allocated = torch.cuda.memory_allocated() / 1e9
        reserved = torch.cuda.memory_reserved() / 1e9
        print(f"    [{label}] GPU: {allocated:.2f}GB alloc / {reserved:.2f}GB reserved")
    
    def extract_weight(linear_layer, expected_shape, keep_on_cpu=True):
        """
        Extract weight from a layer.
        """
        expected_numel = torch.Size(expected_shape).numel() if isinstance(expected_shape, (tuple, list)) else expected_shape.numel()
        
        weight = linear_layer.weight.data.to(compute_dtype)
        
        if weight.shape != expected_shape:
            if weight.numel() == expected_numel:
                weight = weight.reshape(expected_shape)
            elif len(weight.shape) == 2 and len(expected_shape) == 2:
                if weight.shape[0] == expected_shape[1] and weight.shape[1] == expected_shape[0]:
                    weight = weight.t()
                elif weight.numel() == expected_numel:
                    weight = weight.reshape(expected_shape)
        
        return weight.cpu() if keep_on_cpu else weight.cuda()
    
    total_layers = len(model.model.layers)
    if layer_indices is None:
        target_layers = list(range(total_layers))
    else:
        target_layers = layer_indices
    print(f"\n  Processing {len(target_layers)} layers: {target_layers[:5]}{'...' if len(target_layers) > 5 else ''}\n")
    
    for i in target_layers:
        layer = model.model.layers[i]
        original_mlp = layer.mlp
        
        ram = check_ram()
        print(f"  Layer {i+1}/{total_layers} (RAM: {ram:.1f}%):")
        
        print(f"    Extracting weights to CPU...", end=" ", flush=True)
        with torch.no_grad():
            gate_w_full = extract_weight(original_mlp.gate_proj, (config.intermediate_size, hidden_size), keep_on_cpu=True)
            up_w_full = extract_weight(original_mlp.up_proj, (config.intermediate_size, hidden_size), keep_on_cpu=True)
            down_w_full = extract_weight(original_mlp.down_proj, (hidden_size, config.intermediate_size), keep_on_cpu=True)
            
            gate_w = gate_w_full[:intermediate_size, :].clone()
            up_w = up_w_full[:intermediate_size, :].clone()
            down_w = down_w_full[:, :intermediate_size].clone()
            
            del gate_w_full, up_w_full, down_w_full
            gc.collect()
        print("Done")
        
        print(f"    Deleting original MLP...", end=" ", flush=True)
        del original_mlp
        layer.mlp = None
        cleanup(aggressive=True)
        print("Done")
        
        print(f"    Creating MoE layer...", end=" ", flush=True)
        moe_layer = MoELayer(
            hidden_size=hidden_size,
            intermediate_size=intermediate_size,
            num_experts=num_experts,
            num_experts_per_tok=num_experts_per_tok,
            router_jitter_noise=router_jitter_noise,
            router_aux_loss_coef=router_aux_loss_coef,
            bnb_config=bnb_config,
            device='cpu' if bnb_config is None else device,
            init_on_cpu=(bnb_config is None),
            dtype=compute_dtype,
            enable_cpu_offload=enable_cpu_offload
        )
        print("Done")
        
        print(f"    Copying weights to {num_experts} experts ...", end=" ", flush=True)
        with torch.no_grad():
            gate_target_shape = moe_layer.gate_proj[0].weight.shape
            up_target_shape = moe_layer.up_proj[0].weight.shape
            down_target_shape = moe_layer.down_proj[0].weight.shape
            
            if gate_w.shape != gate_target_shape:
                if gate_w.numel() == gate_target_shape.numel():
                    gate_w = gate_w.reshape(gate_target_shape)
                else:
                    gate_w = gate_w[:gate_target_shape[0], :gate_target_shape[1]]
            
            if up_w.shape != up_target_shape:
                if up_w.numel() == up_target_shape.numel():
                    up_w = up_w.reshape(up_target_shape)
                else:
                    up_w = up_w[:up_target_shape[0], :up_target_shape[1]]
            
            if down_w.shape != down_target_shape:
                if down_w.numel() == down_target_shape.numel():
                    down_w = down_w.reshape(down_target_shape)
                elif len(down_w.shape) == 1 or (len(down_w.shape) == 2 and down_w.shape[1] == 1):
                    if down_w.numel() == down_target_shape.numel():
                        down_w = down_w.reshape(down_target_shape)
                    else:
                        transposed_shape = (down_target_shape[1], down_target_shape[0])
                        if down_w.numel() == torch.Size(transposed_shape).numel():
                            down_w = down_w.reshape(transposed_shape).t()
                        else:
                            raise RuntimeError(
                                f"Cannot reshape down_w from {down_w.shape} to {down_target_shape}. "
                                f"down_w.numel()={down_w.numel()}, target.numel()={down_target_shape.numel()}."
                            )
                elif down_w.shape[0] == down_target_shape[0] and down_w.shape[1] >= down_target_shape[1]:
                    down_w = down_w[:, :down_target_shape[1]]
                elif down_w.shape[1] == down_target_shape[1] and down_w.shape[0] >= down_target_shape[0]:
                    down_w = down_w[:down_target_shape[0], :]
                elif down_w.shape[0] == down_target_shape[1] and down_w.shape[1] == down_target_shape[0]:
                    down_w = down_w.t()
                else:
                    raise RuntimeError(
                        f"Cannot reshape down_w from {down_w.shape} to {down_target_shape}. "
                        f"down_w.numel()={down_w.numel()}, target.numel()={down_target_shape.numel()}."
                    )
            
            for idx in range(num_experts):
                moe_layer.gate_proj[idx].weight.copy_(gate_w)
                moe_layer.up_proj[idx].weight.copy_(up_w)
                moe_layer.down_proj[idx].weight.copy_(down_w)
        
        print("Done")
        
        del gate_w, up_w, down_w
        gc.collect()
        
        print(f"    Moving to GPU...", end=" ", flush=True)
        moe_layer.gate_proj = moe_layer.gate_proj.to(device)
        moe_layer.up_proj = moe_layer.up_proj.to(device)
        moe_layer.down_proj = moe_layer.down_proj.to(device)
        moe_layer.router = moe_layer.router.to(device)
        print("Done")
        
        layer.mlp = moe_layer
        cleanup(aggressive=True)
        
        if (i + 1) % 4 == 0:
            print_memory_stats(f"Layer {i+1}")
    
    cleanup(aggressive=True)
    
    print(f"\n Successfully replaced {len(target_layers)} FFN layers with MoE")
    print(f"  Expert dispatch: Efficient sparse routing (top-{num_experts_per_tok})")
    print(f"  Params per expert: ~{(intermediate_size * hidden_size * 3) / 1e6:.1f}M")
    print(f"  Active params per token: ~{(intermediate_size * hidden_size * 3 * num_experts_per_tok) / 1e6:.1f}M")
    print(f"   All experts start identical (no noise)")
    print(f"   Router initialized for top-2 routing (experts 0 & 1 preferred)")
    
    gpu_final = torch.cuda.memory_allocated() / 1e9
    ram_final = check_ram()
    print(f"  Final: GPU {gpu_final:.2f}GB | RAM {ram_final:.1f}%")
    return model

def compute_moe_auxiliary_loss(model, router_aux_loss_coef=0.001):
    """
    Compute auxiliary load balancing loss from all MoE layers.
    L_aux = sum(D_i * P_i) where:
    - D_i = expert frequency (actual utilization rate)
    - P_i = router probability (router confidence)

    Args:
        model: MoE model
        router_aux_loss_coef: Coefficient for auxiliary loss (λ)

    Returns:
        aux_loss: Auxiliary loss value
    """
    total_aux_loss = torch.tensor(0.0, device=next(model.parameters()).device)

    for layer in model.model.layers:
        if hasattr(layer.mlp, '_last_router_probs') and layer.mlp._last_router_probs is not None:
            router_probs = layer.mlp._last_router_probs

            expert_freq = router_probs.mean(dim=0)

            router_confidence = router_probs.mean(dim=0)

            layer_aux_loss = torch.sum(expert_freq * router_confidence) * layer.mlp.num_experts
            total_aux_loss = total_aux_loss + layer_aux_loss

    return total_aux_loss

def compute_moe_loss(model, outputs, router_aux_loss_coef=0.001):
    """
    Compute total loss for MoE model: L_total = L_NTP + λ * L_aux

    Args:
        model: MoE model
        outputs: Model outputs with loss
        router_aux_loss_coef: Coefficient for auxiliary loss (λ)

    Returns:
        total_loss: Combined loss
        ntp_loss: Next token prediction loss
        aux_loss: Auxiliary load balancing loss
    """
    ntp_loss = outputs.loss if hasattr(outputs, 'loss') else None

    aux_loss = compute_moe_auxiliary_loss(model, router_aux_loss_coef)

    if ntp_loss is not None:
        total_loss = ntp_loss + router_aux_loss_coef * aux_loss
    else:
        total_loss = router_aux_loss_coef * aux_loss

    return total_loss, ntp_loss, aux_loss

print("Auxiliary loss computation function defined")

Auxiliary loss computation function defined


In [19]:
import gc
import torch
from transformers import BitsAndBytesConfig, AutoModelForCausalLM
import ctypes

NUM_EXPERTS = 8
NUM_EXPERTS_PER_TOK = 2
ROUTER_JITTER_NOISE = 0.0
ROUTER_AUX_LOSS_COEF = 0.001
USE_HALF_WIDTH = False

print("MoE Model Configuration")

intermediate = 14336 // 2 if USE_HALF_WIDTH else 14336
expert_params = NUM_EXPERTS * 32 * 3 * 4096 * intermediate
expert_memory_bf16 = expert_params * 2 / 1e9
expert_memory_8bit = expert_params * 1 / 1e9
expert_memory_4bit = expert_params * 0.5 / 1e9
print(f"\nMemory Estimate:")
print(f"  Intermediate size: {intermediate} {'(half-width)' if USE_HALF_WIDTH else '(full)'}")
print(f"  Expert params: {expert_params/1e9:.2f}B")
print(f"  Expert memory (bf16): ~{expert_memory_bf16:.1f} GB")
print(f"  Expert memory (8-bit): ~{expert_memory_8bit:.1f} GB")
print(f"  Expert memory (4-bit): ~{expert_memory_4bit:.1f} GB")

print("MEMORY CLEANUP BEFORE MOE CREATION")

gpu_initial = torch.cuda.memory_allocated() / 1e9
print(f"GPU memory before cleanup: {gpu_initial:.2f} GB")

if 'base_model' in globals() and 'teacher_model' in globals():
    if base_model is not teacher_model:
        print("  Deleting base_model (keeping teacher_model for KD)")
        del base_model
    else:
        print("  base_model IS teacher_model, keeping it")
elif 'base_model' in globals():
    print("  Deleting base_model")
    del base_model

vars_to_delete = ['model']
deleted_count = 0
for var_name in list(globals().keys()):
    if var_name in vars_to_delete and var_name in globals():
        obj = globals().get(var_name)
        if obj is not None and hasattr(obj, 'parameters'):
            try:
                next(obj.parameters())
                print(f"  Deleting: {var_name}")
                del globals()[var_name]
                deleted_count += 1
            except:
                pass

for _ in range(3):
    gc.collect()
    torch.cuda.empty_cache()
torch.cuda.synchronize()

try:
    ctypes.CDLL("libc.so.6").malloc_trim(0)
except:
    pass

gpu_after_cleanup = torch.cuda.memory_allocated() / 1e9
print(f"\nGPU memory after cleanup: {gpu_after_cleanup:.2f} GB")
print(f"Freed: {gpu_initial - gpu_after_cleanup:.2f} GB")

moe_expert_bnb_config = BitsAndBytesConfig(
    load_in_8bit=True,
    bnb_8bit_compute_dtype=torch.bfloat16,
    bnb_8bit_use_double_quant=True,
    bnb_8bit_quant_type="nf4",
)

print("Loading fresh Mistral-7B model in bf16 (NO quantization for correct weight extraction)...")
moe_model = AutoModelForCausalLM.from_pretrained(
    model_id,
    device_map="auto",
    trust_remote_code=True,
    torch_dtype=torch.bfloat16,
    low_cpu_mem_usage=True,
)

gpu_with_base = torch.cuda.memory_allocated() / 1e9
print(f" Fresh model loaded")
print(f"  GPU memory: {gpu_with_base:.2f} GB")

print(f"Creating MoE model...")
print(f"  - {NUM_EXPERTS} experts, top-{NUM_EXPERTS_PER_TOK} routing")
print(f"  - Expert precision: bf16 (for correct weight copying)")
print(f"  - Intermediate size: {intermediate}")

moe_model = replace_ffn_with_moe(
    model=moe_model,
    num_experts=NUM_EXPERTS,
    num_experts_per_tok=NUM_EXPERTS_PER_TOK,
    router_jitter_noise=ROUTER_JITTER_NOISE,
    router_aux_loss_coef=ROUTER_AUX_LOSS_COEF,
    bnb_config=None,
    ram_threshold=80.0,
    use_disk_offload=True,
    half_width=USE_HALF_WIDTH,
    enable_cpu_offload=False,
)

gc.collect()
torch.cuda.empty_cache()
torch.cuda.synchronize()

gpu_final = torch.cuda.memory_allocated() / 1e9
gpu_reserved = torch.cuda.memory_reserved() / 1e9
gpu_total = torch.cuda.get_device_properties(0).total_memory / 1e9

print(" MOE MODEL CREATED SUCCESSFULLY")

print(f"  GPU Memory:")
print(f"    Allocated: {gpu_final:.2f} GB")
print(f"    Reserved:  {gpu_reserved:.2f} GB")
print(f"    Total:     {gpu_total:.2f} GB")
print(f"    Usage:     {100*gpu_final/gpu_total:.1f}%")

MoE Model Configuration

Memory Estimate:
  Intermediate size: 14336 (full)
  Expert params: 45.10B
  Expert memory (bf16): ~90.2 GB
  Expert memory (8-bit): ~45.1 GB
  Expert memory (4-bit): ~22.5 GB
MEMORY CLEANUP BEFORE MOE CREATION
GPU memory before cleanup: 7.55 GB
  base_model IS teacher_model, keeping it
  Deleting: model

GPU memory after cleanup: 7.55 GB
Freed: 0.00 GB
Loading fresh Mistral-7B model in bf16 (NO quantization for correct weight extraction)...


`torch_dtype` is deprecated! Use `dtype` instead!


Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

 Fresh model loaded
  GPU memory: 22.03 GB
Creating MoE model...
  - 8 experts, top-2 routing
  - Expert precision: bf16 (for correct weight copying)
  - Intermediate size: 14336
Model configuration:
  Hidden size: 4096
  Original intermediate size: 14336
  MoE intermediate size: 14336
  Number of layers: 32
  Experts per layer: 8
  Experts per token: 2

 Using GPU for weight processing
  GPU Memory: 150.11 GB
  Compute dtype: torch.bfloat16

  Processing 32 layers: [0, 1, 2, 3, 4]...

  Layer 1/32 (RAM: 12.5%):
    Extracting weights to CPU... Done
    Deleting original MLP... Done
    Creating MoE layer... Done
    Copying weights to 8 experts ... Done
    Moving to GPU... Done
  Layer 2/32 (RAM: 12.6%):
    Extracting weights to CPU... Done
    Deleting original MLP... Done
    Creating MoE layer... Done
    Copying weights to 8 experts ... Done
    Moving to GPU... Done
  Layer 3/32 (RAM: 12.2%):
    Extracting weights to CPU... Done
    Deleting original MLP... Done
    Creating M

## Pretraining evaluation for dense MoE

In [20]:
moe_baseline_comprehensive = evaluate_model_comprehensive(
    model=moe_model,
    tokenizer=tokenizer,
    eval_dataset=eval_dataset,
    answer_tokens=ANSWER_TOKENS,
    model_name="moe_baseline",
    device="cuda",
    max_samples_mmlu=1000,
    max_samples_throughput=100,
    show_progress=True
)

print("Note: This is the TRUE baseline - no LoRA adapters applied yet")
print("   Next step: Apply QLoRA for training")


COMPREHENSIVE EVALUATION: moe_baseline

Running MMLU evaluation (n=1000)...


Evaluating: 100%|██████████| 1000/1000 [00:44<00:00, 22.58it/s]



Computing FLOPs...
Measuring throughput (n=100)...
Analyzing parameter efficiency...

COMPREHENSIVE METRICS: moe_baseline
Accuracy Metrics:
  MMLU Accuracy: 0.6640
  Top-2 Accuracy: 0.8310
  ECE: 0.0816

 Computational Efficiency:
  FLOPs per forward pass: 3023.66G
  Tokens/second: 6569.86
  ms/token: 0.15
  Samples/second: 23.10

 Parameter Efficiency:
  Total parameters: 46.70B
  Active parameters: 46.70B
  Trainable parameters: 46702.79M
  Sparsity ratio: 100.00%

 Memory Usage:
  Model size: 89078.51 MB
  GPU allocated: 94.02 GB
  GPU reserved: 94.29 GB

Results saved to: results/moe_baseline_comprehensive.json

Note: This is the TRUE baseline - no LoRA adapters applied yet


## MoE Trainer Classes

In [43]:
from transformers import TrainingArguments, Trainer, DataCollatorForLanguageModeling
from torch.utils.data import Dataset as TorchDataset
import torch

class MoETrainer(Trainer):
    """
    Custom Trainer for MoE models that computes auxiliary load balancing loss.
    Works with both regular models and PEFT-wrapped models.
    Total loss formula: L_total = L_NTP + λ * L_aux
    """

    def __init__(self, router_aux_loss_coef=0.001, *args, **kwargs):
        super().__init__(*args, **kwargs)
        self.router_aux_loss_coef = router_aux_loss_coef

    def _moe_layers(self, model):
        """Get MoE layers from model, handling PEFT wrapper."""
        # Try different model structures (PEFT wrapped, base model, etc.)
        candidates = [
            lambda m: m.model.layers if hasattr(m, 'model') and hasattr(m.model, 'layers') else None,
            lambda m: m.base_model.model.layers if hasattr(m, 'base_model') and hasattr(m.base_model, 'model') and hasattr(m.base_model.model, 'layers') else None,
            lambda m: m.base_model.model.model.layers if hasattr(m, 'base_model') and hasattr(m.base_model, 'model') and hasattr(m.base_model.model, 'model') else None,
        ]
        
        for get_layers in candidates:
            try:
                layers = get_layers(model)
                if layers is not None:
                    return layers
            except:
                continue
        return []

    def compute_loss(self, model, inputs, return_outputs=False, num_items_in_batch=None):
        # Enable router logits collection
        for layer in self._moe_layers(model):
            mlp = getattr(layer, 'mlp', None)
            if mlp is not None and hasattr(mlp, 'forward'):
                mlp._collect_router_logits = True

        # Forward pass
        outputs = model(**inputs)
        ntp_loss = outputs.loss if hasattr(outputs, 'loss') else outputs[0]
        
        # Compute auxiliary load balancing loss
        aux_loss = self._compute_moe_auxiliary_loss(model)
        total_loss = ntp_loss + self.router_aux_loss_coef * aux_loss

        # Log losses periodically
        if self.state.global_step % self.args.logging_steps == 0:
            self.log({
                "train/ntp_loss": ntp_loss.item(),
                "train/aux_loss": aux_loss.item(),
                "train/total_loss": total_loss.item(),
                "train/aux_loss_weight": self.router_aux_loss_coef,
            })

        # Disable router logits collection
        for layer in self._moe_layers(model):
            mlp = getattr(layer, 'mlp', None)
            if mlp is not None and hasattr(mlp, '_collect_router_logits'):
                mlp._collect_router_logits = False

        return (total_loss, outputs) if return_outputs else total_loss

    def _compute_moe_auxiliary_loss(self, model):
        """Compute auxiliary loss from all MoE layers."""
        device = next(model.parameters()).device
        aux_loss = torch.tensor(0.0, device=device)

        for layer in self._moe_layers(model):
            mlp = getattr(layer, 'mlp', None)
            if mlp is not None and hasattr(mlp, 'compute_auxiliary_loss'):
                aux_loss = aux_loss + mlp.compute_auxiliary_loss()
        return aux_loss

# Training configuration for QLoRA
TRAINING_CONFIG = {
    "num_experts": NUM_EXPERTS,
    "num_experts_per_tok": NUM_EXPERTS_PER_TOK,
    "router_jitter_noise": ROUTER_JITTER_NOISE,
    "router_aux_loss_coef": ROUTER_AUX_LOSS_COEF,
    "learning_rate": 2e-4,  # QLoRA standard learning rate
    "batch_size": 4,  # Larger batch with QLoRA
    "gradient_accumulation_steps": 4,
    "warmup_ratio": 0.03,
    "num_train_epochs": 1,
    "logging_steps": 25,
    "eval_steps": 100,
    "save_steps": 100,
    "output_dir": "./mistral_moe_qlora",
}

print(" MoETrainer class defined (QLoRA compatible)\n")

 MoETrainer class defined (QLoRA compatible)



In [44]:
import torch

print("MoETrainer with Knowledge Distillation class defined\n")
# ============================================================================
# 15.5.2B - INTEGRATED MOE KD TRAINER
# ============================================================================

class IntegratedMoEKDTrainer(Trainer):
    """
    KD for MoE: L_total = (1-α)*L_NTP + α*L_KD + λ*L_aux + β*L_routingKD
    """
    def __init__(self, teacher_model=None, kd_config=None, router_aux_loss_coef=0.001, *args, **kwargs):
        super().__init__(*args, **kwargs)
        self.teacher_model = teacher_model
        self.kd_config = kd_config or {}
        self.router_aux_loss_coef = router_aux_loss_coef
        self.kd_alpha = self.kd_config.get('kd_alpha', 0.5)
        self.temperature = self.kd_config.get('temperature', 4.0)
        self.routing_kd_weight = self.kd_config.get('routing_kd_weight', 0.0)
        self.enable_routing_kd = self.kd_config.get('enable_routing_kd', False)

    def _moe_layers(self, model):
        if hasattr(model, 'model') and hasattr(model.model, 'layers'):
            return model.model.layers
        if hasattr(model, 'base_model') and hasattr(model.base_model.model, 'layers'):
            return model.base_model.model.layers
        return []

    def compute_loss(self, model, inputs, return_outputs=False, num_items_in_batch=None):
        """
        Compute loss with KD.
        NOTE: Trainer automatically handles device placement - inputs are already on correct device!
        """
        for layer in self._moe_layers(model):
            mlp = getattr(layer, 'mlp', None)
            if mlp is not None and hasattr(mlp, 'forward'):
                mlp._collect_router_logits = True

        outputs = model(**inputs)
        ntp_loss = outputs.loss if hasattr(outputs, 'loss') else outputs[0]
        device = ntp_loss.device if hasattr(ntp_loss, 'device') else next(model.parameters()).device

        kd_loss = torch.tensor(0.0, device=device)
        routing_kd_loss = torch.tensor(0.0, device=device)

        if self.teacher_model is not None and self.kd_alpha > 0:
            tdev = next(self.teacher_model.parameters()).device
            tinputs = {k: (v.to(tdev) if hasattr(v, 'to') else v) for k, v in inputs.items()}
            with torch.no_grad():
                t_out = self.teacher_model(**tinputs)

            s_logits = outputs.logits / self.temperature
            t_logits = t_out.logits / self.temperature

            kd_loss = F.kl_div(
                F.log_softmax(s_logits, dim=-1),
                F.softmax(t_logits, dim=-1),
                reduction='batchmean'
            ) * (self.temperature ** 2)

            if self.enable_routing_kd and self.routing_kd_weight > 0:
                ent = torch.tensor(0.0, device=device)
                layers_count = 0
                for layer in self._moe_layers(model):
                    mlp = getattr(layer, 'mlp', None)
                    if mlp is not None and hasattr(mlp, '_last_router_probs') and mlp._last_router_probs is not None:
                        probs = mlp._last_router_probs
                        ent_layer = -(probs * torch.log(probs + 1e-10)).sum(dim=-1).mean()
                        ent = ent + ent_layer
                        layers_count += 1
                if layers_count > 0:
                    routing_kd_loss = ent / layers_count

        aux_loss = torch.tensor(0.0, device=device)
        for layer in self._moe_layers(model):
            mlp = getattr(layer, 'mlp', None)
            if mlp is not None and hasattr(mlp, 'compute_auxiliary_loss'):
                aux_loss = aux_loss + mlp.compute_auxiliary_loss()

        total_loss = ((1 - self.kd_alpha) * ntp_loss
                      + self.kd_alpha * kd_loss
                      + self.router_aux_loss_coef * aux_loss
                      + self.routing_kd_weight * routing_kd_loss)

        for layer in self._moe_layers(model):
            mlp = getattr(layer, 'mlp', None)
            if mlp is not None and hasattr(mlp, '_collect_router_logits'):
                mlp._collect_router_logits = False

        return (total_loss, outputs) if return_outputs else total_loss

print(" IntegratedMoEKDTrainer class defined")

MoETrainer with Knowledge Distillation class defined

 IntegratedMoEKDTrainer class defined


## MoE Model Standard Training Setup

In [45]:
import torch
from transformers import TrainingArguments, DataCollatorForLanguageModeling
from peft import get_peft_model, LoraConfig, TaskType, prepare_model_for_kbit_training
import os

# Training mode selection
USE_KNOWLEDGE_DISTILLATION = False  # Set to True for KD, False for standard training

# Dataset subset configuration
USE_SUBSET = True           # Use only a subset of the data
SUBSET_PERCENTAGE = 0.2     # Use 20% of the dataset

# MoE and training hyperparameters
TRAINING_CONFIG = {
    "num_experts": NUM_EXPERTS,
    "num_experts_per_tok": NUM_EXPERTS_PER_TOK,
    "router_jitter_noise": ROUTER_JITTER_NOISE,
    "router_aux_loss_coef": 0.01,  
    "learning_rate": 2e-4,
    "batch_size": 4,
    "gradient_accumulation_steps": 4,
    "warmup_ratio": 0.1,  
    "num_train_epochs": 1,
    "logging_steps": 25,
    "eval_steps": 50,
    "save_steps": 100,
    "max_steps": 250,  # Train for 250 steps
    "max_length": 512,
}

# LoRA configuration - INCLUDES ROUTERS
LORA_CONFIG = {
    "r": 16,
    "lora_alpha": 32,
    "lora_dropout": 0.05,
}

# Knowledge distillation configuration
KD_CONFIG = {
    'kd_alpha': 0.5,
    'temperature': 4.0,
    'routing_kd_weight': 0.0,
    'enable_routing_kd': False,
    'router_aux_loss_coef': TRAINING_CONFIG['router_aux_loss_coef'],
    'name': 'Standard KD',
}

# Output directory
OUTPUT_DIR = "./mistral_moe_qlora_kd" if USE_KNOWLEDGE_DISTILLATION else "./mistral_moe_qlora"


In [46]:
# --------------------------------------------------------------------------
# 3. SETUP MODELS
# --------------------------------------------------------------------------

student_model = moe_model

student_model.gradient_checkpointing_enable()
student_model.enable_input_require_grads()

print("Student model prepared for training")


Student model prepared for training


In [47]:
# Tokenize the dataset for train/validation splits
# IMPORTANT: Training uses prompts WITH answers, evaluation uses prompts WITHOUT answers

def tokenize_function_for_training(examples):
    """Tokenize WITH answers for training - model learns to predict correct answer."""
    return tokenizer(
        examples['formatted_prompt_with_answer'],  # <-- WITH ANSWER
        truncation=True,
        max_length=TRAINING_CONFIG['max_length'],
        padding=False,
    )

def tokenize_function_for_eval(examples):
    """Tokenize WITHOUT answers for evaluation loss computation."""
    return tokenizer(
        examples['formatted_prompt'],  # <-- WITHOUT ANSWER
        truncation=True,
        max_length=TRAINING_CONFIG['max_length'],
        padding=False,
    )

print("Tokenizing dataset...")
print("  Training data: using prompts WITH answers (teaches model correct answers)")
print("  Eval data: using prompts WITHOUT answers (for loss computation)")

# Tokenize training data WITH answers
tokenized_train = dataset['train'].map(
    tokenize_function_for_training,
    batched=True,
    remove_columns=dataset['train'].column_names,
    desc="Tokenizing training data (with answers)"
)

# Tokenize eval data WITHOUT answers (for loss computation during training)
tokenized_eval = dataset['test'].map(
    tokenize_function_for_eval,
    batched=True,
    remove_columns=dataset['test'].column_names,
    desc="Tokenizing eval data (without answers)"
)

print("Tokenization complete.")
print(f"  Training samples: {len(tokenized_train):,}")
print(f"  Eval samples: {len(tokenized_eval):,}")

if USE_SUBSET:
    num_train_samples = int(len(tokenized_train) * SUBSET_PERCENTAGE)
    num_val_samples = int(len(tokenized_eval) * SUBSET_PERCENTAGE)
    
    train_dataset = tokenized_train.select(range(num_train_samples))
    val_dataset = tokenized_eval.select(range(num_val_samples))
    
    print(f"\nUsing {SUBSET_PERCENTAGE*100}% subset of data:")
    print(f"  Train samples: {len(train_dataset)} (with answers)")
    print(f"  Validation samples: {len(val_dataset)} (without answers)")
else:
    train_dataset = tokenized_train
    val_dataset = tokenized_eval
    print(f"\nUsing full dataset:")
    print(f"  Train samples: {len(train_dataset)} (with answers)")
    print(f"  Validation samples: {len(val_dataset)} (without answers)")

Tokenizing dataset...
  Training data: using prompts WITH answers (teaches model correct answers)
  Eval data: using prompts WITHOUT answers (for loss computation)


Tokenizing training data (with answers):   0%|          | 0/68940 [00:00<?, ? examples/s]

Tokenizing eval data (without answers):   0%|          | 0/29547 [00:00<?, ? examples/s]

Tokenization complete.
  Training samples: 68,940
  Eval samples: 29,547

Using 20.0% subset of data:
  Train samples: 13788 (with answers)
  Validation samples: 5909 (without answers)


In [48]:
from peft import get_peft_model, LoraConfig, TaskType

# --------------------------------------
# 8. BUILD TARGET MODULES LIST FOR LORA 
# -------------------------------------

print("\nBuilding target modules list for LoRA...")

# Standard attention/MLP modules
target_modules = [
    "q_proj", "k_proj", "v_proj", "o_proj",  # Attention
]


# --------------------------------------------------------------------------
# 9. APPLY LORA TO STUDENT MODEL
# --------------------------------------------------------------------------

print("\nApplying LoRA to student model...")

peft_config = LoraConfig(
    task_type=TaskType.CAUSAL_LM,
    inference_mode=False,
    r=LORA_CONFIG['r'],
    lora_alpha=LORA_CONFIG['lora_alpha'],
    lora_dropout=LORA_CONFIG['lora_dropout'],
    target_modules=target_modules,
    bias="none",
)

student_model = get_peft_model(student_model, peft_config)

print("LoRA applied successfully!")
print(f"  LoRA rank (r): {LORA_CONFIG['r']}")
print(f"  LoRA alpha: {LORA_CONFIG['lora_alpha']}")
print(f"  LoRA dropout: {LORA_CONFIG['lora_dropout']}")

# --------------------------------------------------------------------------
# 10. VERIFY TRAINABLE PARAMETERS
# --------------------------------------------------------------------------

print("\nTrainable parameter summary:")
student_model.print_trainable_parameters()

# Detailed breakdown
total_params = sum(p.numel() for p in student_model.parameters())
trainable_params = sum(p.numel() for p in student_model.parameters() if p.requires_grad)
frozen_params = total_params - trainable_params

print(f"\nDetailed parameter breakdown:")
print(f"  Total parameters: {total_params:,}")
print(f"  Trainable parameters: {trainable_params:,} ({100 * trainable_params / total_params:.2f}%)")
print(f"  Frozen parameters: {frozen_params:,} ({100 * frozen_params / total_params:.2f}%)")






Building target modules list for LoRA...

Applying LoRA to student model...
LoRA applied successfully!
  LoRA rank (r): 16
  LoRA alpha: 32
  LoRA dropout: 0.05

Trainable parameter summary:
trainable params: 13,631,488 || all params: 46,716,424,448 || trainable%: 0.0292

Detailed parameter breakdown:
  Total parameters: 46,716,424,448
  Trainable parameters: 13,631,488 (0.03%)
  Frozen parameters: 46,702,792,960 (99.97%)


In [49]:
# --------------------------------------------------------------------------
# 11. SETUP TRAINING ARGUMENTS
# --------------------------------------------------------------------------

print("\nSetting up training arguments...")

output_dir = "./moe-mistral-mmlu-lora"
os.makedirs(output_dir, exist_ok=True)

training_args = TrainingArguments(
    output_dir=output_dir,
    num_train_epochs=TRAINING_CONFIG['num_train_epochs'],
    max_steps=TRAINING_CONFIG['max_steps'],
    per_device_train_batch_size=TRAINING_CONFIG['batch_size'],
    per_device_eval_batch_size=TRAINING_CONFIG['batch_size'],
    gradient_accumulation_steps=TRAINING_CONFIG['gradient_accumulation_steps'],
    learning_rate=TRAINING_CONFIG['learning_rate'],
    warmup_ratio=TRAINING_CONFIG['warmup_ratio'],
    logging_dir=f"{output_dir}/logs",
    logging_steps=TRAINING_CONFIG['logging_steps'],
    eval_strategy="steps",
    eval_steps=TRAINING_CONFIG['eval_steps'],
    save_strategy="steps",
    save_steps=TRAINING_CONFIG['save_steps'],
    save_total_limit=2,
    load_best_model_at_end=True,
    metric_for_best_model="eval_loss",
    greater_is_better=False,
    fp16=False,
    bf16=True,
    gradient_checkpointing=True,
    optim="paged_adamw_8bit",
    report_to=["tensorboard", "wandb"],
    remove_unused_columns=False,
    dataloader_pin_memory=True,
    dataloader_num_workers=2,
)

print("Training arguments configured:")
print(f"  Output directory: {output_dir}")
print(f"  Max steps: {TRAINING_CONFIG['max_steps']}")
print(f"  Batch size: {TRAINING_CONFIG['batch_size']}")
print(f"  Gradient accumulation: {TRAINING_CONFIG['gradient_accumulation_steps']}")
print(f"  Effective batch size: {TRAINING_CONFIG['batch_size'] * TRAINING_CONFIG['gradient_accumulation_steps']}")
print(f"  Learning rate: {TRAINING_CONFIG['learning_rate']}")
print(f"  Warmup ratio: {TRAINING_CONFIG['warmup_ratio']}")



Setting up training arguments...
Training arguments configured:
  Output directory: ./moe-mistral-mmlu-lora
  Max steps: 250
  Batch size: 4
  Gradient accumulation: 4
  Effective batch size: 16
  Learning rate: 0.0002
  Warmup ratio: 0.1


## Execute MoE Model Standard Training

In [50]:
from transformers import DataCollatorForLanguageModeling
data_collator = DataCollatorForLanguageModeling(tokenizer=tokenizer, mlm=False)
print("\nInitializing trainer...")

eval_dataset_for_accuracy = eval_dataset  # Raw dataset (has 'formatted_prompt', 'answer')
eval_tokenized_for_loss = val_dataset     # Tokenized dataset (has 'input_ids', 'attention_mask')

# Select trainer based on configuration
if USE_KNOWLEDGE_DISTILLATION:
    print("  Using MoEKDTrainer for knowledge distillation")
    # CORRECT - passes KD_CONFIG as a dictionary
    trainer = IntegratedMoEKDTrainer(
        model=student_model,
        args=training_args,
        train_dataset=train_dataset,
        eval_dataset=val_dataset,
        data_collator=data_collator,
        teacher_model=teacher_model,
        kd_config=KD_CONFIG,                     
        router_aux_loss_coef=TRAINING_CONFIG['router_aux_loss_coef'],
        callbacks=[
            ComprehensiveEvalCallback(
                eval_dataset_for_accuracy=eval_dataset_for_accuracy,
                eval_tokenized_for_loss=eval_tokenized_for_loss,
                tokenizer=tokenizer,
                answer_tokens=ANSWER_TOKENS,
                eval_steps=TRAINING_CONFIG['eval_steps'],
                accuracy_samples=1000,
                device="cuda"
            )
        ],
    )
else:
    print("Using MoETrainer for standard training")
    trainer = MoETrainer(
        model=student_model,
        args=training_args,
        train_dataset=train_dataset,
        eval_dataset=val_dataset,
        data_collator=data_collator,
        router_aux_loss_coef=TRAINING_CONFIG['router_aux_loss_coef'],
        callbacks=[
            ComprehensiveEvalCallback(
                eval_dataset_for_accuracy=eval_dataset_for_accuracy,
                eval_tokenized_for_loss=eval_tokenized_for_loss,
                tokenizer=tokenizer,
                answer_tokens=ANSWER_TOKENS,
                eval_steps=TRAINING_CONFIG['eval_steps'],
                accuracy_samples=1000,
                device="cuda"
            )
        ],
    )

print("Trainer initialized successfully(false for standard training)!")

The model is already on multiple devices. Skipping the move to device specified in `args`.



Initializing trainer...
Using MoETrainer for standard training
Trainer initialized successfully(false for standard training)!


In [ ]:
import torch
import gc

print("\n" + "=" * 80)
print("STARTING MOE MODEL STANDARD TRAINING")
print("=" * 80)
print("Note: No evaluation during training - will evaluate after training completes")

torch.cuda.empty_cache()
gc.collect()

train_result_moe_standard = trainer_moe_standard.train()

print("\n" + "=" * 80)
print("MOE MODEL STANDARD TRAINING COMPLETE")
print("=" * 80)

print("\nFinal training metrics:")
for key, value in train_result_moe_standard.metrics.items():
    if isinstance(value, float):
        print(f"  {key}: {value:.4f}")
    else:
        print(f"  {key}: {value}")

print(f"\nSaving model to {OUTPUT_DIR_MOE}/final_model...")
trainer_moe_standard.save_model(f"{OUTPUT_DIR_MOE}/final_model")
tokenizer.save_pretrained(f"{OUTPUT_DIR_MOE}/final_model")


STARTING MOE MODEL STANDARD TRAINING


Step,Training Loss
25,33.067200


KeyboardInterrupt: 

: 

## MoE Standard Training Model Evaluation

In [42]:
import json
import os
import numpy as np
import wandb

student_model_moe.eval()

moe_standard_comprehensive = evaluate_model_comprehensive(
    model=student_model_moe,
    tokenizer=tokenizer,
    eval_dataset=eval_dataset,
    answer_tokens=ANSWER_TOKENS,
    model_name="moe_standard",
    device="cuda",
    max_samples_mmlu=1000,
    max_samples_throughput=100,
    show_progress=True
)

try:
    if wandb.run is not None:
        wandb.log({
            'moe_standard/accuracy': moe_standard_comprehensive['accuracy'],
            'moe_standard/top2_accuracy': moe_standard_comprehensive['top2_accuracy'],
            'moe_standard/ece': moe_standard_comprehensive['ece'],
            'moe_standard/flops_billions': moe_standard_comprehensive['flops']/1e9,
            'moe_standard/tokens_per_second': moe_standard_comprehensive['tokens_per_second'],
            'moe_standard/ms_per_token': moe_standard_comprehensive['ms_per_token'],
            'moe_standard/active_params_billions': moe_standard_comprehensive['active_params']/1e9,
            'moe_standard/gpu_memory_gb': moe_standard_comprehensive['gpu_memory_allocated_gb'],
        })
except:
    pass

print("\nPost-training evaluation complete!")
print("Results saved to results/moe_standard_comprehensive.json")


COMPREHENSIVE EVALUATION: moe_standard

Running MMLU evaluation (n=1000)...


Evaluating: 100%|██████████| 1000/1000 [00:52<00:00, 18.88it/s]



Computing FLOPs...
Measuring throughput (n=100)...
Analyzing parameter efficiency...

COMPREHENSIVE METRICS: moe_standard
Accuracy Metrics:
  MMLU Accuracy: 0.2350
  Top-2 Accuracy: 0.4580
  ECE: 0.2065

 Computational Efficiency:
  FLOPs per forward pass: 3023.66G
  Tokens/second: 5301.29
  ms/token: 0.19
  Samples/second: 18.64

 Parameter Efficiency:
  Total parameters: 46.72B
  Active parameters: 46.72B
  Trainable parameters: 13.63M
  Sparsity ratio: 100.00%

 Memory Usage:
  Model size: 89130.51 MB
  GPU allocated: 95.42 GB
  GPU reserved: 97.25 GB

Results saved to: results/moe_standard_comprehensive.json


Post-training evaluation complete!
Results saved to results/moe_standard_comprehensive.json


## MoE Model Knowledge Distillation Training Setup

In [ ]:
from transformers import TrainingArguments, DataCollatorForLanguageModeling
from peft import get_peft_model, LoraConfig, TaskType, prepare_model_for_kbit_training
import os
import torch

USE_KNOWLEDGE_DISTILLATION_MOE = True

USE_SUBSET_MOE_KD = True
SUBSET_PERCENTAGE_MOE_KD = 0.2

TRAINING_CONFIG_MOE_KD = {
    "num_experts": NUM_EXPERTS,
    "num_experts_per_tok": NUM_EXPERTS_PER_TOK,
    "router_jitter_noise": ROUTER_JITTER_NOISE,
    "router_aux_loss_coef": 0.01,
    "learning_rate": 2e-4,
    "batch_size": 4,
    "gradient_accumulation_steps": 4,
    "warmup_ratio": 0.1,
    "num_train_epochs": 1,
    "logging_steps": 25,
    "eval_steps": 50,
    "save_steps": 100,
    "max_steps": 250,
    "max_length": 512,
}

LORA_CONFIG_MOE_KD = {
    "r": 16,
    "lora_alpha": 32,
    "lora_dropout": 0.05,
    "target_modules": ["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
    "task_type": TaskType.CAUSAL_LM,
}

KD_CONFIG_MOE = {
    'kd_alpha': 0.5,
    'temperature': 4.0,
    'routing_kd_weight': 0.0,
    'enable_routing_kd': False,
    'router_aux_loss_coef': TRAINING_CONFIG_MOE_KD['router_aux_loss_coef'],
    'name': 'Standard KD',
}

OUTPUT_DIR_MOE_KD = "./mistral_moe_qlora_kd"

print("=" * 80)
print("MOE MODEL KNOWLEDGE DISTILLATION TRAINING CONFIGURATION")
print("=" * 80)
print(f"Training Mode: Knowledge Distillation")
print(f"KD Alpha: {KD_CONFIG_MOE['kd_alpha']}")
print(f"Temperature: {KD_CONFIG_MOE['temperature']}")
print(f"Output Directory: {OUTPUT_DIR_MOE_KD}")
print(f"Max Steps: {TRAINING_CONFIG_MOE_KD['max_steps']}")
print("=" * 80)

if 'teacher_model' not in globals() or teacher_model is None:
    if 'base_model' in globals():
        teacher_model = base_model.eval()
        for param in teacher_model.parameters():
            param.requires_grad = False
        print("\nTeacher model set from base_model")
    else:
        raise ValueError("teacher_model or base_model must be available!")

print("\nSetting up student model for MoE KD...")

student_model_moe_kd = moe_model

student_model_moe_kd.gradient_checkpointing_enable()
student_model_moe_kd.enable_input_require_grads()

if 'tokenized_train' not in globals():
    def tokenize_function_for_training(examples):
        return tokenizer(
            examples['formatted_prompt_with_answer'],
            truncation=True,
            max_length=TRAINING_CONFIG_MOE_KD['max_length'],
            padding=False,
        )
    
    tokenized_train = dataset['train'].map(
        tokenize_function_for_training,
        batched=True,
        remove_columns=dataset['train'].column_names,
        desc="Tokenizing training data"
    )

if 'train_dataset' in globals() and train_dataset is not None:
    base_train_dataset_moe_kd = train_dataset
else:
    base_train_dataset_moe_kd = tokenized_train

if USE_SUBSET_MOE_KD:
    subset_size = int(len(base_train_dataset_moe_kd) * SUBSET_PERCENTAGE_MOE_KD)
    train_dataset_subset_moe_kd = base_train_dataset_moe_kd.select(range(subset_size))
    print(f"Using {SUBSET_PERCENTAGE_MOE_KD*100}% of training data: {len(train_dataset_subset_moe_kd)} samples")
else:
    train_dataset_subset_moe_kd = base_train_dataset_moe_kd

training_args_moe_kd = TrainingArguments(
    output_dir=OUTPUT_DIR_MOE_KD,
    per_device_train_batch_size=TRAINING_CONFIG_MOE_KD['batch_size'],
    per_device_eval_batch_size=TRAINING_CONFIG_MOE_KD['batch_size'],
    gradient_accumulation_steps=TRAINING_CONFIG_MOE_KD['gradient_accumulation_steps'],
    learning_rate=TRAINING_CONFIG_MOE_KD['learning_rate'],
    warmup_ratio=TRAINING_CONFIG_MOE_KD['warmup_ratio'],
    num_train_epochs=TRAINING_CONFIG_MOE_KD['num_train_epochs'],
    logging_steps=TRAINING_CONFIG_MOE_KD['logging_steps'],
    save_steps=TRAINING_CONFIG_MOE_KD['save_steps'],
    max_steps=TRAINING_CONFIG_MOE_KD['max_steps'],
    eval_strategy="no",
    save_strategy="steps",
    load_best_model_at_end=False,
    fp16=False,
    bf16=True,
    gradient_checkpointing=True,
    optim="paged_adamw_8bit",
    report_to="wandb" if 'wandb' in globals() else None,
    remove_unused_columns=False,
    dataloader_pin_memory=True,
    dataloader_num_workers=2,
)

data_collator_moe_kd = DataCollatorForLanguageModeling(tokenizer=tokenizer, mlm=False)

print("\nInitializing trainer...")
print("  Using IntegratedMoEKDTrainer for knowledge distillation")
print("  No evaluation callback - evaluation will be done post-training only")

trainer_moe_kd = IntegratedMoEKDTrainer(
    model=student_model_moe_kd,
    args=training_args_moe_kd,
    train_dataset=train_dataset_subset_moe_kd,
    eval_dataset=None,
    data_collator=data_collator_moe_kd,
    teacher_model=teacher_model,
    kd_config=KD_CONFIG_MOE,
    router_aux_loss_coef=TRAINING_CONFIG_MOE_KD['router_aux_loss_coef'],
)

print("Trainer initialized successfully!")

## Execute MoE Model Knowledge Distillation Training

In [ ]:
import torch
import gc

print("\n" + "=" * 80)
print("STARTING MOE MODEL KNOWLEDGE DISTILLATION TRAINING")
print("=" * 80)
print("Note: No evaluation during training - will evaluate after training completes")

torch.cuda.empty_cache()
gc.collect()

train_result_moe_kd = trainer_moe_kd.train()

print("\n" + "=" * 80)
print("MOE MODEL KNOWLEDGE DISTILLATION TRAINING COMPLETE")
print("=" * 80)

print("\nFinal training metrics:")
for key, value in train_result_moe_kd.metrics.items():
    if isinstance(value, float):
        print(f"  {key}: {value:.4f}")
    else:
        print(f"  {key}: {value}")

print(f"\nSaving model to {OUTPUT_DIR_MOE_KD}/final_model...")
trainer_moe_kd.save_model(f"{OUTPUT_DIR_MOE_KD}/final_model")
tokenizer.save_pretrained(f"{OUTPUT_DIR_MOE_KD}/final_model")

## MoE Knowledge Distillation Model Evaluation

In [ ]:
import json
import os
import numpy as np
import wandb

student_model_moe_kd.eval()

moe_kd_comprehensive = evaluate_model_comprehensive(
    model=student_model_moe_kd,
    tokenizer=tokenizer,
    eval_dataset=eval_dataset,
    answer_tokens=ANSWER_TOKENS,
    model_name="moe_kd",
    device="cuda",
    max_samples_mmlu=1000,
    max_samples_throughput=100,
    show_progress=True
)

try:
    if wandb.run is not None:
        wandb.log({
            'moe_kd/accuracy': moe_kd_comprehensive['accuracy'],
            'moe_kd/top2_accuracy': moe_kd_comprehensive['top2_accuracy'],
            'moe_kd/ece': moe_kd_comprehensive['ece'],
            'moe_kd/flops_billions': moe_kd_comprehensive['flops']/1e9,
            'moe_kd/tokens_per_second': moe_kd_comprehensive['tokens_per_second'],
            'moe_kd/ms_per_token': moe_kd_comprehensive['ms_per_token'],
            'moe_kd/active_params_billions': moe_kd_comprehensive['active_params']/1e9,
            'moe_kd/gpu_memory_gb': moe_kd_comprehensive['gpu_memory_allocated_gb'],
        })
except:
    pass

print("\nPost-training evaluation complete!")
print("Results saved to results/moe_kd_comprehensive.json")

## MoE Routing Variants Configuration

In [ ]:
from typing import Literal, Optional, List
import json

class MoEExperimentConfig:
    """Configuration for MoE variant experiments."""

    num_experts: int = 8
    num_experts_per_tok: int = 2

    router_jitter_noise: float = 0.0
    router_aux_loss_coef: float = 0.001

    expert_layers: Literal["all", "every_2", "every_4", "selected"] = "all"
    layer_indices: Optional[List[int]] = None

    load_balancing_loss_coef: float = 0.01

    experiment_name: str = "default"
    description: str = ""

    def to_dict(self):
        return {
            'num_experts': self.num_experts,
            'num_experts_per_tok': self.num_experts_per_tok,
            'router_jitter_noise': self.router_jitter_noise,
            'router_aux_loss_coef': self.router_aux_loss_coef,
            'expert_layers': self.expert_layers,
            'layer_indices': self.layer_indices,
            'load_balancing_loss_coef': self.load_balancing_loss_coef,
            'experiment_name': self.experiment_name,
            'description': self.description,
        }

    def save(self, filepath):
        with open(filepath, 'w') as f:
            json.dump(self.to_dict(), f, indent=2)

    @classmethod
    def load(cls, filepath):
        with open(filepath, 'r') as f:
            return cls(**json.load(f))

EXPERIMENT_CONFIGS = {
    "moe_baseline": MoEExperimentConfig(
        num_experts=8,
        num_experts_per_tok=2,
        router_jitter_noise=0.0,
        router_aux_loss_coef=0.001,
        expert_layers="all",
        experiment_name="moe_baseline",
        description="Baseline MoE: 8 experts, top-2 routing, all layers, standard config"
    ),
    
    "top1_8x1": MoEExperimentConfig(
        num_experts=8,
        num_experts_per_tok=1,
        expert_layers="all",
        experiment_name="top1_8x1",
        description="Top-1 routing: 8 experts, single expert per token"
    ),

    "top1_16x1": MoEExperimentConfig(
        num_experts=16,
        num_experts_per_tok=1,
        expert_layers="all",
        experiment_name="top1_16x1",
        description="Top-1 routing: 16 experts, single expert per token"
    ),

    "routing_noisy_8x2": MoEExperimentConfig(
        num_experts=8,
        num_experts_per_tok=2,
        router_jitter_noise=0.2,
        expert_layers="all",
        experiment_name="routing_noisy_8x2",
        description="Noisy routing: 8 experts, top-2, high jitter (0.2)"
    ),

    "balanced_8x2": MoEExperimentConfig(
        num_experts=8,
        num_experts_per_tok=2,
        router_aux_loss_coef=0.05,
        expert_layers="all",
        experiment_name="balanced_8x2",
        description="Load Balanced: 8 experts, top-2, high aux loss coef (0.05)"
    ),

    "efficient_4x1": MoEExperimentConfig(
        num_experts=4,
        num_experts_per_tok=1,
        expert_layers="all",
        experiment_name="efficient_4x1",
        description="Efficient: 4 experts, top-1 routing"
    ),

    "large_16x2": MoEExperimentConfig(
        num_experts=16,
        num_experts_per_tok=2,
        expert_layers="all",
        experiment_name="large_16x2",
        description="Large: 16 experts, top-2 routing"
    ),

    "sparse_8x2": MoEExperimentConfig(
        num_experts=8,
        num_experts_per_tok=2,
        expert_layers="every_2",
        experiment_name="sparse_8x2",
        description="Sparse placement: experts every 2nd layer"
    ),

    "placement_early_8x2": MoEExperimentConfig(
        num_experts=8,
        num_experts_per_tok=2,
        expert_layers="selected",
        layer_indices=list(range(0, 16)),
        experiment_name="placement_early_8x2",
        description="Early placement: Experts in first 16 layers only"
    ),

    "placement_middle_8x2": MoEExperimentConfig(
        num_experts=8,
        num_experts_per_tok=2,
        expert_layers="selected",
        layer_indices=list(range(8, 24)),
        experiment_name="placement_middle_8x2",
        description="Middle placement: Experts in middle 16 layers (8-23)"
    ),

    "placement_late_8x2": MoEExperimentConfig(
        num_experts=8,
        num_experts_per_tok=2,
        expert_layers="selected",
        layer_indices=list(range(16, 32)),
        experiment_name="placement_late_8x2",
        description="Late placement: Experts in last 16 layers only"
    ),

    "placement_mixed_8x2": MoEExperimentConfig(
        num_experts=8,
        num_experts_per_tok=2,
        expert_layers="selected",
        layer_indices=[0, 1, 2, 3, 14, 15, 16, 17, 28, 29, 30, 31],
        experiment_name="placement_mixed_8x2",
        description="Mixed placement: Experts in first 4, middle 4, and last 4 layers"
    ),

    "baseline_8x2": MoEExperimentConfig(
        num_experts=8,
        num_experts_per_tok=2,
        expert_layers="all",
        experiment_name="baseline_8x2",
        description="Baseline: 8 experts, top-2 routing, all layers"
    ),
}

print("MoE variant configuration system defined")
print(f"  Available configs: {list(EXPERIMENT_CONFIGS.keys())}")

## MoE Experiment Runner

In [ ]:
import gc
import torch
import time
import os
import json
import numpy as np
from transformers import TrainingArguments, DataCollatorForLanguageModeling, AutoModelForCausalLM
from peft import get_peft_model, LoraConfig, TaskType, prepare_model_for_kbit_training

class MoEExperimentRunner:
    """System for running and tracking MoE variant experiments."""

    def __init__(self, base_model, tokenizer, eval_dataset, answer_tokens, train_dataset=None):
        self.base_model = base_model
        self.tokenizer = tokenizer
        self.eval_dataset = eval_dataset
        self.answer_tokens = answer_tokens
        self.train_dataset = train_dataset
        self.results = {}

    def _clear_all_models_except_teacher(self):
        """
        Aggressively clear ALL models from GPU memory except teacher_model.
        """
        print("  Clearing all models from memory (preserving teacher_model)...")
        
        teacher_model = None
        if 'teacher_model' in globals():
            teacher_model = globals()['teacher_model']
        
        model_vars = ['moe_model', 'student_model', 'base_model', 'model']
        
        for var_name in model_vars:
            if var_name in globals():
                var = globals()[var_name]
                if var is not teacher_model and var is not None:
                    try:
                        if hasattr(var, 'cpu'):
                            var.cpu()
                        elif hasattr(var, 'to'):
                            var.to('cpu')
                    except:
                        pass
                    
                    try:
                        if hasattr(var, 'parameters'):
                            for param in list(var.parameters()):
                                if param is not None:
                                    del param
                        if hasattr(var, 'buffers'):
                            for buffer in list(var.buffers()):
                                if buffer is not None:
                                    del buffer
                    except:
                        pass
                    
                    try:
                        del globals()[var_name]
                        print(f"    Deleted {var_name} from globals")
                    except:
                        pass
        
        if hasattr(self, 'current_model') and self.current_model is not None:
            if self.current_model is not teacher_model:
                try:
                    if hasattr(self.current_model, 'cpu'):
                        self.current_model.cpu()
                except:
                    pass
                del self.current_model
        
        if torch.cuda.is_available():
            torch.cuda.empty_cache()
            torch.cuda.synchronize()
            torch.cuda.empty_cache()
            torch.cuda.ipc_collect()
        
        for _ in range(3):
            gc.collect()
        
        if teacher_model is not None:
            globals()['teacher_model'] = teacher_model
        
        if torch.cuda.is_available():
            allocated = torch.cuda.memory_allocated() / 1e9
            reserved = torch.cuda.memory_reserved() / 1e9
            print(f"    GPU Memory after global cleanup: {allocated:.2f} GB allocated, {reserved:.2f} GB reserved")

    def _cleanup_model(self, model, preserve_teacher=True):
        """
        Aggressively clean up a model from GPU memory.
        """
        if model is None:
            return
        
        teacher_model = None
        if preserve_teacher and 'teacher_model' in globals():
            teacher_model = globals()['teacher_model']
        
        if model is teacher_model:
            print("  Skipping cleanup of teacher_model (preserved for KD)")
            return
        
        try:
            if hasattr(model, 'cpu'):
                model.cpu()
            elif hasattr(model, 'to'):
                model.to('cpu')
        except Exception as e:
            print(f"  Warning: Could not move model to CPU: {e}")
        
        try:
            if hasattr(model, 'parameters'):
                for param in list(model.parameters()):
                    if param is not None:
                        del param
            if hasattr(model, 'buffers'):
                for buffer in list(model.buffers()):
                    if buffer is not None:
                        del buffer
        except Exception as e:
            print(f"  Warning: Error deleting parameters: {e}")
        
        try:
            del model
        except Exception as e:
            print(f"  Warning: Error deleting model: {e}")
        
        if torch.cuda.is_available():
            torch.cuda.empty_cache()
            torch.cuda.synchronize()
            torch.cuda.empty_cache()
            torch.cuda.ipc_collect()
        
        for _ in range(2):
            gc.collect()
        
        if preserve_teacher and teacher_model is not None:
            globals()['teacher_model'] = teacher_model
        
        print("  Model cleanup completed")

    def _evaluate_comprehensive(self, model, config, max_samples):
        """
        Run comprehensive evaluation using standardized function.
        """
        model_name = f"{config.experiment_name}_pretrain"
        
        results = evaluate_model_comprehensive(
            model=model,
            tokenizer=self.tokenizer,
            eval_dataset=self.eval_dataset,
            answer_tokens=self.answer_tokens,
            model_name=model_name,
            device="cuda",
            max_samples_mmlu=max_samples,
            max_samples_throughput=100,
            show_progress=False
        )
        
        return results

    def _create_moe_model(self, config):
        """Create MoE model according to configuration."""
        print("  Clearing memory before loading model...")
        
        teacher_model = None
        if 'teacher_model' in globals():
            teacher_model = globals()['teacher_model']
            print(f"  Preserving teacher_model in memory (for KD)")
        
        if torch.cuda.is_available():
            torch.cuda.empty_cache()
            torch.cuda.synchronize()
            torch.cuda.empty_cache()
            torch.cuda.ipc_collect()
        
        for _ in range(2):
            gc.collect()
        
        if teacher_model is not None:
            globals()['teacher_model'] = teacher_model

        model = AutoModelForCausalLM.from_pretrained(
            model_id,
            device_map="auto",
            trust_remote_code=True,
            torch_dtype=torch.bfloat16,
            low_cpu_mem_usage=True,
        )

        total_layers = len(model.model.layers)
        if config.expert_layers == "all":
            target_layers = None
        elif config.expert_layers == "every_2":
            target_layers = list(range(0, total_layers, 2))
        elif config.expert_layers == "every_4":
            target_layers = list(range(0, total_layers, 4))
        elif config.expert_layers == "selected" and config.layer_indices:
            target_layers = config.layer_indices
        else:
            target_layers = None

        model = replace_ffn_with_moe(
            model=model,
            num_experts=config.num_experts,
            num_experts_per_tok=config.num_experts_per_tok,
            router_jitter_noise=config.router_jitter_noise,
            router_aux_loss_coef=config.router_aux_loss_coef,
            bnb_config=None,
            ram_threshold=80.0,
            use_disk_offload=True,
            layer_indices=target_layers,
            half_width=False,
            enable_cpu_offload=False,
        )

        torch.cuda.empty_cache()
        return model

    def _apply_lora(self, model):
        """Apply LoRA adapters for training."""
        is_quantized = False
        try:
            for name, module in model.named_modules():
                if hasattr(module, 'weight') and hasattr(module.weight, 'quant_state'):
                    is_quantized = True
                    break
        except:
            pass
        
        if is_quantized:
            model = prepare_model_for_kbit_training(model)
        else:
            model.gradient_checkpointing_enable()
            model.enable_input_require_grads()

        lora_config = LoraConfig(
            r=16, lora_alpha=32,
            target_modules=["q_proj", "k_proj", "v_proj", "o_proj"],
            lora_dropout=0.05, bias="none",
            task_type=TaskType.CAUSAL_LM
        )

        model = get_peft_model(model, lora_config)
        model.train()
        return model

    def _train_model(self, model, config, steps, training_mode="standard", kd_config=None, skip_eval=True):
        """
        Run training loop with support for standard or KD training.
        """
        teacher_model = None
        if training_mode == "kd":
            if 'teacher_model' in globals():
                teacher_model = globals()['teacher_model']
            else:
                raise ValueError("teacher_model not found in globals() for KD training")

        eval_dataset_for_trainer = None
        if not skip_eval and self.eval_dataset is not None:
            if hasattr(self.eval_dataset, 'column_names') and 'input_ids' in self.eval_dataset.column_names:
                eval_dataset_for_trainer = self.eval_dataset
            else:
                def tokenize_eval_function(examples):
                    return self.tokenizer(
                        examples['formatted_prompt'],
                        truncation=True,
                        max_length=512,
                        padding=False,
                    )
                
                eval_dataset_for_trainer = self.eval_dataset.map(
                    tokenize_eval_function,
                    batched=True,
                    remove_columns=self.eval_dataset.column_names,
                    desc="Tokenizing eval dataset for Trainer"
                )
        elif skip_eval:
            print("  Skipping training evaluation for faster training")
        
        output_dir = f"experiments/{config.experiment_name}_{training_mode}_checkpoints"
        os.makedirs(output_dir, exist_ok=True)

        args = TrainingArguments(
            output_dir=output_dir,
            max_steps=steps,
            per_device_train_batch_size=4,
            gradient_accumulation_steps=4,
            learning_rate=2e-4,
            warmup_ratio=0.1,
            logging_dir=f"{output_dir}/logs",
            logging_steps=25,
            eval_strategy="no",
            save_strategy="steps",
            save_steps=steps,
            save_total_limit=1,
            load_best_model_at_end=False,
            fp16=False,
            bf16=True,
            gradient_checkpointing=True,
            optim="paged_adamw_8bit",
            report_to=["tensorboard", "wandb"],
            remove_unused_columns=False,
            dataloader_pin_memory=True,
            dataloader_num_workers=2,
        )

        if training_mode == "kd":
            print(f"  Using IntegratedMoEKDTrainer for knowledge distillation")
            print(f"  KD Config: alpha={kd_config.get('kd_alpha', 0.5)}, temp={kd_config.get('temperature', 4.0)}, routing_kd={kd_config.get('enable_routing_kd', False)}")
            
            trainer_kwargs = {
                "model": model,
                "args": args,
                "train_dataset": self.train_dataset,
                "tokenizer": self.tokenizer,
                "data_collator": DataCollatorForLanguageModeling(self.tokenizer, mlm=False),
                "teacher_model": teacher_model,
                "kd_config": kd_config,
                "router_aux_loss_coef": config.router_aux_loss_coef
            }
            if not skip_eval and eval_dataset_for_trainer is not None:
                trainer_kwargs["eval_dataset"] = eval_dataset_for_trainer
            
            trainer = IntegratedMoEKDTrainer(**trainer_kwargs)
        else:
            print(f"  Using MoETrainer for standard training")
            trainer_kwargs = {
                "model": model,
                "args": args,
                "train_dataset": self.train_dataset,
                "tokenizer": self.tokenizer,
                "data_collator": DataCollatorForLanguageModeling(self.tokenizer, mlm=False),
                "router_aux_loss_coef": config.router_aux_loss_coef
            }
            if not skip_eval and eval_dataset_for_trainer is not None:
                trainer_kwargs["eval_dataset"] = eval_dataset_for_trainer
            
            trainer = MoETrainer(**trainer_kwargs)

        trainer.train()

    def _save_results(self, exp_name, results):
        """Save results to disk."""
        filepath = f"experiments/{exp_name}_results.json"
        os.makedirs("experiments", exist_ok=True)

        def make_serializable(obj):
            if isinstance(obj, np.ndarray):
                return obj.tolist()
            if isinstance(obj, np.float32):
                return float(obj)
            return str(obj)

        clean_results = {k: (v if not isinstance(v, (np.ndarray, set)) else str(v)) for k, v in results.items()}

        with open(filepath, 'w') as f:
            json.dump(clean_results, f, indent=2, default=make_serializable)

    def run_experiment(self, config: MoEExperimentConfig, max_samples=1000, train=False, 
                      train_steps=250, training_mode="standard", kd_config=None,
                      skip_training_eval=True):
        """
        Run a complete experiment with given configuration.
        """
        print(f"\n{'='*70}")
        print(f"RUNNING EXPERIMENT: {config.experiment_name}")
        print(f"{'='*70}\n")
        print(f"Description: {config.description}")
        print(f"Configuration:")
        print(f"  Experts: {config.num_experts}")
        print(f"  Experts per token: {config.num_experts_per_tok}")
        print(f"  Layer placement: {config.expert_layers}")
        if train:
            print(f"  Training mode: {training_mode}")
            print(f"  Training steps: {train_steps}")
        print()

        print("Cleaning up memory before experiment...")
        self._clear_all_models_except_teacher()
        
        if torch.cuda.is_available():
            torch.cuda.empty_cache()
            torch.cuda.synchronize()
        gc.collect()
        
        if torch.cuda.is_available():
            allocated = torch.cuda.memory_allocated() / 1e9
            reserved = torch.cuda.memory_reserved() / 1e9
            print(f"  GPU Memory before model creation: {allocated:.2f} GB allocated, {reserved:.2f} GB reserved")
            if allocated > 10.0:
                print(f"  WARNING: High GPU memory usage detected ({allocated:.2f} GB).")
                print(f"  This may cause OOM. Teacher model may be using this memory.")

        moe_model = self._create_moe_model(config)
        self.current_model = moe_model

        print(f"\nPhase 1: Pre-training Evaluation (n={max_samples})...")
        pre_results = self._evaluate_comprehensive(
            model=moe_model,
            config=config,
            max_samples=max_samples
        )

        final_results = pre_results.copy()
        final_results['phase'] = 'pre_train_only'
        final_results['training_mode'] = training_mode if train else None

        if train and self.train_dataset:
            print(f"\nPhase 2: Training for {train_steps} steps ({training_mode} mode)...")
            
            if training_mode == "kd":
                if kd_config is None:
                    raise ValueError("kd_config must be provided when training_mode='kd'")
                if 'teacher_model' not in globals():
                    raise ValueError("teacher_model must exist in globals() for KD training")

            moe_model = self._apply_lora(moe_model)
            self.current_model = moe_model
            self._train_model(
                moe_model, 
                config, 
                train_steps, 
                training_mode=training_mode,
                kd_config=kd_config,
                skip_eval=skip_training_eval
            )

            print(f"\nPhase 3: Post-training Evaluation (n={max_samples})...")
            post_results = self._evaluate_comprehensive(
                model=moe_model,
                config=config,
                max_samples=max_samples
            )

            final_results = post_results.copy()
            final_results['phase'] = 'trained'
            final_results['training_mode'] = training_mode
            final_results['pre_train_accuracy'] = pre_results['accuracy']
            final_results['accuracy_gain'] = post_results['accuracy'] - pre_results['accuracy']
            final_results['pre_train_results'] = pre_results

        final_results['config'] = config.to_dict()
        final_results['timestamp'] = time.time()

        self.results[config.experiment_name] = final_results
        self._save_results(config.experiment_name, final_results)

        print(f"\nCleaning up {config.experiment_name} model...")
        self._cleanup_model(moe_model, preserve_teacher=True)
        
        if hasattr(self, 'current_model'):
            self.current_model = None
        
        if torch.cuda.is_available():
            allocated = torch.cuda.memory_allocated() / 1e9
            reserved = torch.cuda.memory_reserved() / 1e9
            print(f"  GPU Memory after cleanup: {allocated:.2f} GB allocated, {reserved:.2f} GB reserved")
            teacher_preserved = 'teacher_model' in globals()
            print(f"  Teacher model preserved: {teacher_preserved}")

        return final_results

print("MoEExperimentRunner class defined")

## MoE Variant Experiments (Standard Training)

In [ ]:
import wandb
import traceback
import gc
import torch

RUN_VARIANT_EXPERIMENTS_STANDARD = True

if RUN_VARIANT_EXPERIMENTS_STANDARD:
    print("RUNNING MOE VARIANT EXPERIMENTS (STANDARD TRAINING)")

    torch.cuda.empty_cache()
    gc.collect()

    runner_standard = MoEExperimentRunner(
        base_model=None,
        tokenizer=tokenizer,
        eval_dataset=eval_dataset,
        answer_tokens=ANSWER_TOKENS,
        train_dataset=train_dataset_subset_moe if 'train_dataset_subset_moe' in globals() else tokenized_train
    )

    experiments_to_run_standard = [
        "efficient_4x1",
        "baseline_8x2",
        "top1_8x1",
        "sparse_8x2",
        "placement_early_8x2",
        "placement_middle_8x2",
        "placement_late_8x2",
        "routing_noisy_8x2",
        "moe_baseline",
        "balanced_8x2",
        "placement_mixed_8x2",
    ]

    for exp_name in experiments_to_run_standard:
        if exp_name in EXPERIMENT_CONFIGS:
            config = EXPERIMENT_CONFIGS[exp_name]

            try:
                results = runner_standard.run_experiment(
                    config, 
                    max_samples=1000, 
                    train=True, 
                    train_steps=250,
                    training_mode="standard",
                    skip_training_eval=True
                )

                if wandb.run is not None:
                    wandb.log({
                        f'variants_standard/{exp_name}/accuracy': results['accuracy'],
                        f'variants_standard/{exp_name}/tokens_per_second': results['tokens_per_second'],
                        f'variants_standard/{exp_name}/flops': results['flops']/1e9,
                    })

            except Exception as e:
                print(f"Error in experiment {exp_name}: {str(e)}")
                traceback.print_exc()
                continue

    print("\nVariant experiments (standard training) complete!")

## MoE Variant Experiments (Knowledge Distillation)

In [ ]:
import wandb
import traceback
import gc
import torch

RUN_VARIANT_EXPERIMENTS_KD = True

if 'teacher_model' not in globals() or teacher_model is None:
    print("ERROR: teacher_model not found. Please run the teacher model setup cell first.")
else:
    print(f"Teacher model ready for KD training")
    print(f"  Teacher model device: {next(teacher_model.parameters()).device}")

if RUN_VARIANT_EXPERIMENTS_KD and 'teacher_model' in globals() and teacher_model is not None:
    print("RUNNING MOE VARIANT EXPERIMENTS (KNOWLEDGE DISTILLATION)")

    torch.cuda.empty_cache()
    gc.collect()

    runner_kd = MoEExperimentRunner(
        base_model=None,
        tokenizer=tokenizer,
        eval_dataset=eval_dataset,
        answer_tokens=ANSWER_TOKENS,
        train_dataset=train_dataset_subset_moe_kd if 'train_dataset_subset_moe_kd' in globals() else tokenized_train
    )

    experiments_to_run_kd = [
        "efficient_4x1",
        "baseline_8x2",
        "top1_8x1",
        "sparse_8x2",
        "placement_early_8x2",
        "placement_middle_8x2",
        "placement_late_8x2",
        "routing_noisy_8x2",
        "moe_baseline",
        "balanced_8x2",
        "placement_mixed_8x2",
    ]

    for exp_name in experiments_to_run_kd:
        if exp_name in EXPERIMENT_CONFIGS:
            config = EXPERIMENT_CONFIGS[exp_name]

            try:
                results = runner_kd.run_experiment(
                    config, 
                    max_samples=1000, 
                    train=True, 
                    train_steps=250,
                    training_mode="kd",
                    kd_config=KD_CONFIG_STANDARD,
                    skip_training_eval=True
                )

                if wandb.run is not None:
                    wandb.log({
                        f'variants_kd/{exp_name}/accuracy': results['accuracy'],
                        f'variants_kd/{exp_name}/tokens_per_second': results['tokens_per_second'],
                        f'variants_kd/{exp_name}/flops': results['flops']/1e9,
                    })

            except Exception as e:
                print(f"Error in experiment {exp_name}: {str(e)}")
                traceback.print_exc()
                continue

    print("\nVariant experiments (knowledge distillation) complete!")

## Unified Model Comparison Visualization

In [ ]:
import json
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

def format_value(value, format_type='float', decimals=4):
    """Format value based on type."""
    if value is None:
        return "N/A"
    
    if format_type == 'float':
        return f"{value:.{decimals}f}"
    elif format_type == 'percent':
        return f"{value*100:.2f}%"
    elif format_type == 'billions':
        return f"{value/1e9:.2f}B"
    elif format_type == 'millions':
        return f"{value/1e6:.2f}M"
    else:
        return str(value)

print("=" * 80)
print("LOADING MODEL RESULTS FOR UNIFIED COMPARISON")
print("=" * 80)

dense_baseline = {}
if os.path.exists("results/baseline_comprehensive.json"):
    try:
        with open("results/baseline_comprehensive.json", 'r') as f:
            dense_baseline = json.load(f)
        print("Loaded Dense Baseline from results/baseline_comprehensive.json")
    except:
        pass

if not dense_baseline or 'accuracy' not in dense_baseline:
    if 'baseline_comprehensive' in globals():
        dense_baseline = baseline_comprehensive
        print("Using in-memory Dense Baseline values")
    else:
        print("Dense Baseline not available")

dense_kd = {}
if os.path.exists("results/dense_kd_comprehensive.json"):
    try:
        with open("results/dense_kd_comprehensive.json", 'r') as f:
            dense_kd = json.load(f)
        print("Loaded Dense with KD from results/dense_kd_comprehensive.json")
    except:
        pass

if not dense_kd or 'accuracy' not in dense_kd:
    if 'kd_dense_comprehensive' in globals():
        dense_kd = kd_dense_comprehensive
        print("Using in-memory Dense KD values")
    else:
        print("Dense with KD not yet evaluated")

moe_standard = {}
if os.path.exists("results/moe_standard_comprehensive.json"):
    try:
        with open("results/moe_standard_comprehensive.json", 'r') as f:
            moe_standard = json.load(f)
        print("Loaded MoE Standard from results/moe_standard_comprehensive.json")
    except:
        pass

if not moe_standard or 'accuracy' not in moe_standard:
    if 'moe_standard_comprehensive' in globals():
        moe_standard = moe_standard_comprehensive
        print("Using in-memory MoE Standard values")
    else:
        print("MoE Standard not yet evaluated")

moe_kd = {}
if os.path.exists("results/moe_kd_comprehensive.json"):
    try:
        with open("results/moe_kd_comprehensive.json", 'r') as f:
            moe_kd = json.load(f)
        print("Loaded MoE KD from results/moe_kd_comprehensive.json")
    except:
        pass

if not moe_kd or 'accuracy' not in moe_kd:
    if 'moe_kd_comprehensive' in globals():
        moe_kd = moe_kd_comprehensive
        print("Using in-memory MoE KD values")
    else:
        print("MoE KD not yet evaluated")

print("\n" + "=" * 80)
print("RESULTS LOADED")
print("=" * 80)

unified_data = {
    'Dense Baseline (Standard)': dense_baseline,
    'Dense with KD': dense_kd,
    'MoE Standard Training': moe_standard,
    'MoE Knowledge Distillation': moe_kd,
}

table_data = []

for model_name, metrics in unified_data.items():
    if not metrics or 'accuracy' not in metrics:
        continue
    
    accuracy = metrics.get('accuracy', None)
    top2_accuracy = metrics.get('top2_accuracy', None)
    ece = metrics.get('ece', None)
    throughput = metrics.get('samples_per_second', None)
    latency = metrics.get('ms_per_token', None)
    tokens_per_second = metrics.get('tokens_per_second', None)
    params = metrics.get('total_params', None)
    flops = metrics.get('flops', None)
    gpu_memory = metrics.get('gpu_memory_allocated_gb', None)
    
    table_data.append({
        'Model': model_name,
        'Accuracy': accuracy,
        'Top-2 Accuracy': top2_accuracy,
        'ECE': ece,
        'Throughput (samples/s)': throughput,
        'Latency (ms/token)': latency,
        'Tokens/sec': tokens_per_second,
        'FLOPs (G)': flops / 1e9 if flops is not None else None,
        'Parameters (B)': params / 1e9 if params is not None else None,
        'GPU Memory (GB)': gpu_memory,
    })

df = pd.DataFrame(table_data)

print("\n" + "=" * 150)
print("UNIFIED MODEL COMPARISON TABLE")
print("=" * 150)
print()

formatted_rows = []
header = f"{'Model':<35} {'Accuracy':<12} {'Top-2':<10} {'ECE':<10} {'Throughput':<15} {'Latency':<15} {'Tokens/s':<12} {'FLOPs':<12} {'Params':<12} {'GPU':<12}"
formatted_rows.append(header)
formatted_rows.append("-" * 150)

for _, row in df.iterrows():
    model = row['Model']
    acc = f"{row['Accuracy']:.4f}" if pd.notna(row['Accuracy']) else "N/A"
    top2 = f"{row['Top-2 Accuracy']:.4f}" if pd.notna(row['Top-2 Accuracy']) else "N/A"
    ece = f"{row['ECE']:.4f}" if pd.notna(row['ECE']) else "N/A"
    thr = f"{row['Throughput (samples/s)']:.2f}" if pd.notna(row['Throughput (samples/s)']) else "N/A"
    lat = f"{row['Latency (ms/token)']:.2f}" if pd.notna(row['Latency (ms/token)']) else "N/A"
    tok = f"{row['Tokens/sec']:.2f}" if pd.notna(row['Tokens/sec']) else "N/A"
    flops = f"{row['FLOPs (G)']:.2f}G" if pd.notna(row['FLOPs (G)']) else "N/A"
    params = f"{row['Parameters (B)']:.2f}B" if pd.notna(row['Parameters (B)']) else "N/A"
    gpu = f"{row['GPU Memory (GB)']:.2f}GB" if pd.notna(row['GPU Memory (GB)']) else "N/A"
    
    formatted_rows.append(
        f"{model:<35} {acc:<12} {top2:<10} {ece:<10} {thr:<15} {lat:<15} {tok:<12} {flops:<12} {params:<12} {gpu:<12}"
    )

for row in formatted_rows:
    print(row)

print("=" * 150)
print()

print("Detailed DataFrame:")
print(df.to_string(index=False))
print()

os.makedirs("results", exist_ok=True)

csv_path = "results/unified_model_comparison.csv"
df.to_csv(csv_path, index=False)
print(f"Unified comparison table saved to: {csv_path}")

json_path = "results/unified_model_comparison.json"
with open(json_path, 'w') as f:
    json.dump(unified_data, f, indent=2, default=str)
print(f"Unified comparison data saved to: {json_path}")

table_json_path = "results/unified_model_comparison_table.json"
with open(table_json_path, 'w') as f:
    json.dump(table_data, f, indent=2, default=str)
print(f"Unified table data saved to: {table_json_path}")

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np

sns.set_style("whitegrid")
plt.rcParams['figure.figsize'] = (18, 12)
plt.rcParams['font.size'] = 10

models = df['Model'].values
metrics_to_plot = {
    'Accuracy': df['Accuracy'].values,
    'ECE': df['ECE'].values,
    'Throughput (samples/s)': df['Throughput (samples/s)'].values,
    'Latency (ms/token)': df['Latency (ms/token)'].values,
    'Parameters (B)': df['Parameters (B)'].values,
}

fig, axes = plt.subplots(3, 2, figsize=(16, 14))
fig.suptitle('Unified Model Comparison: Comprehensive Metrics', fontsize=18, fontweight='bold', y=0.995)

axes_flat = axes.flatten()

colors = ['#2E86AB', '#A23B72', '#F18F01', '#C73E1D']

ax1 = axes_flat[0]
valid_idx = [i for i, v in enumerate(metrics_to_plot['Accuracy']) if v is not None]
if valid_idx:
    bars1 = ax1.bar([models[i] for i in valid_idx], 
            [metrics_to_plot['Accuracy'][i] for i in valid_idx],
            color=colors, alpha=0.8, edgecolor='black', linewidth=1.2)
    ax1.set_ylabel('Accuracy', fontsize=12, fontweight='bold')
    ax1.set_title('Accuracy Comparison', fontsize=13, fontweight='bold', pad=10)
    ax1.set_ylim([0, max(metrics_to_plot['Accuracy'][i] for i in valid_idx) * 1.15])
    ax1.grid(axis='y', alpha=0.3, linestyle='--')
    ax1.set_xticklabels([models[i] for i in valid_idx], rotation=45, ha='right', fontsize=9)
    for i, (bar, idx) in enumerate(zip(bars1, valid_idx)):
        height = bar.get_height()
        ax1.text(bar.get_x() + bar.get_width()/2., height,
                f'{height:.3f}',
                ha='center', va='bottom', fontsize=9, fontweight='bold')

ax2 = axes_flat[1]
valid_idx = [i for i, v in enumerate(metrics_to_plot['ECE']) if v is not None]
if valid_idx:
    bars2 = ax2.bar([models[i] for i in valid_idx], 
            [metrics_to_plot['ECE'][i] for i in valid_idx],
            color=colors, alpha=0.8, edgecolor='black', linewidth=1.2)
    ax2.set_ylabel('ECE (Expected Calibration Error)', fontsize=12, fontweight='bold')
    ax2.set_title('Calibration Error Comparison', fontsize=13, fontweight='bold', pad=10)
    ax2.grid(axis='y', alpha=0.3, linestyle='--')
    ax2.set_xticklabels([models[i] for i in valid_idx], rotation=45, ha='right', fontsize=9)
    for i, (bar, idx) in enumerate(zip(bars2, valid_idx)):
        height = bar.get_height()
        ax2.text(bar.get_x() + bar.get_width()/2., height,
                f'{height:.4f}',
                ha='center', va='bottom', fontsize=9, fontweight='bold')
    ax2.text(0.02, 0.98, 'Lower is Better', transform=ax2.transAxes,
            fontsize=9, verticalalignment='top', bbox=dict(boxstyle='round', 
            facecolor='wheat', alpha=0.5))

ax3 = axes_flat[2]
valid_idx = [i for i, v in enumerate(metrics_to_plot['Throughput (samples/s)']) if v is not None]
if valid_idx:
    bars3 = ax3.bar([models[i] for i in valid_idx], 
            [metrics_to_plot['Throughput (samples/s)'][i] for i in valid_idx],
            color=colors, alpha=0.8, edgecolor='black', linewidth=1.2)
    ax3.set_ylabel('Throughput (samples/sec)', fontsize=12, fontweight='bold')
    ax3.set_title('Throughput Comparison', fontsize=13, fontweight='bold', pad=10)
    ax3.grid(axis='y', alpha=0.3, linestyle='--')
    ax3.set_xticklabels([models[i] for i in valid_idx], rotation=45, ha='right', fontsize=9)
    for i, (bar, idx) in enumerate(zip(bars3, valid_idx)):
        height = bar.get_height()
        ax3.text(bar.get_x() + bar.get_width()/2., height,
                f'{height:.2f}',
                ha='center', va='bottom', fontsize=9, fontweight='bold')
    ax3.text(0.02, 0.98, 'Higher is Better', transform=ax3.transAxes,
            fontsize=9, verticalalignment='top', bbox=dict(boxstyle='round', 
            facecolor='lightgreen', alpha=0.5))

ax4 = axes_flat[3]
valid_idx = [i for i, v in enumerate(metrics_to_plot['Latency (ms/token)']) if v is not None]
if valid_idx:
    bars4 = ax4.bar([models[i] for i in valid_idx], 
            [metrics_to_plot['Latency (ms/token)'][i] for i in valid_idx],
            color=colors, alpha=0.8, edgecolor='black', linewidth=1.2)
    ax4.set_ylabel('Latency (ms/token)', fontsize=12, fontweight='bold')
    ax4.set_title('Latency Comparison', fontsize=13, fontweight='bold', pad=10)
    ax4.grid(axis='y', alpha=0.3, linestyle='--')
    ax4.set_xticklabels([models[i] for i in valid_idx], rotation=45, ha='right', fontsize=9)
    for i, (bar, idx) in enumerate(zip(bars4, valid_idx)):
        height = bar.get_height()
        ax4.text(bar.get_x() + bar.get_width()/2., height,
                f'{height:.3f}',
                ha='center', va='bottom', fontsize=9, fontweight='bold')
    ax4.text(0.02, 0.98, 'Lower is Better', transform=ax4.transAxes,
            fontsize=9, verticalalignment='top', bbox=dict(boxstyle='round', 
            facecolor='wheat', alpha=0.5))

ax5 = axes_flat[4]
valid_idx = [i for i, v in enumerate(metrics_to_plot['Parameters (B)']) if v is not None]
if valid_idx:
    bars5 = ax5.bar([models[i] for i in valid_idx], 
            [metrics_to_plot['Parameters (B)'][i] for i in valid_idx],
            color=colors, alpha=0.8, edgecolor='black', linewidth=1.2)
    ax5.set_ylabel('Parameters (Billions)', fontsize=12, fontweight='bold')
    ax5.set_title('Model Size Comparison', fontsize=13, fontweight='bold', pad=10)
    ax5.grid(axis='y', alpha=0.3, linestyle='--')
    ax5.set_xticklabels([models[i] for i in valid_idx], rotation=45, ha='right', fontsize=9)
    for i, (bar, idx) in enumerate(zip(bars5, valid_idx)):
        height = bar.get_height()
        ax5.text(bar.get_x() + bar.get_width()/2., height,
                f'{height:.2f}B',
                ha='center', va='bottom', fontsize=9, fontweight='bold')

axes_flat[5].axis('off')

plt.tight_layout(rect=[0, 0, 1, 0.98])

fig_path = "results/unified_model_comparison_visualization.png"
plt.savefig(fig_path, dpi=300, bbox_inches='tight', facecolor='white')
print(f"Visualization saved to: {fig_path}")

plt.show()